# Data Cleaning

In [1]:
# Import required libraries 
import pandas as pd  
import geopandas as gpd
import geodatasets
import matplotlib.pyplot as plt

In [2]:
# Read dataset 
food = pd.read_csv("../data/raw/food-data.csv")

# View first five lines
food.head()

,Inspection ID,DBA Name,AKA Name,License #,Facility Type,Risk,Address,City,State,Zip,Inspection Date,Inspection Type,Results,Violations,Latitude,Longitude,Location
0,2634806,ONE STOP FOOD & LIQUOR STORE,ONE STOP FOOD & LIQUOR STORE,1094.0,Grocery Store,Risk 2 (Medium),4301-4323 S LAKE PARK AVE,CHICAGO,IL,60653.0,04/10/2026,Complaint,Fail,16. FOOD-CONTACT SURFACES: CLEANED & SANITIZED...,41.816865,-87.598689,"(41.816865148052045, -87.59868884416672)"
1,2634820,MR QUILES MEXICAN FOOD #2,MR QUILES MEXICAN FOOD #2,2385750.0,Mobile Food Preparer,Risk 2 (Medium),2300 S THROOP ST,CHICAGO,IL,60608.0,04/10/2026,Canvass,Pass,47. FOOD & NON-FOOD CONTACT SURFACES CLEANABLE...,41.850451,-87.658798,"(41.85045102427, -87.65879785567869)"
2,2634815,LA GUERITA MEXICAN SHACK,LA GUERITA MEXICAN SHACK,2840820.0,Mobile Food Preparer,Risk 2 (Medium),2300 S THROOP ST,CHICAGO,IL,60608.0,04/10/2026,Canvass,Pass,NaN,41.850451,-87.658798,"(41.85045102427, -87.65879785567869)"
3,2634821,THE BBQ SHACK,THE BBQ SHACK,3065284.0,Restaurant,Risk 1 (High),1709 W WASHINGTON BLVD,CHICAGO,IL,60612.0,04/10/2026,License,Fail,16. FOOD-CONTACT SURFACES: CLEANED & SANITIZED...,41.883143,-87.669767,"(41.88314294811615, -87.66976710838105)"
4,2634811,"Thomas, Velma ECC","Thomas, Velma ECC",26891.0,School,Risk 1 (High),3625 S Hoyne ST,CHICAGO,IL,60609.0,04/10/2026,Canvass,Pass,10. ADEQUATE HANDWASHING SINKS PROPERLY SUPPLI...,41.827769,-87.677505,"(41.82776913597991, -87.67750501083977)"


In [3]:
# Assign variable to represent the number of rows before cleaning
n_before_cleaning = len(food)

print("Number of rows: ", n_before_cleaning)

Number of rows:  308585


In [4]:
# Drop easily accesible duplicate rows based on 'Inspection ID'
food = food.drop_duplicates(subset='Inspection ID')

# Assign variable to represent the number of rows before cleaning
n_before_cleaning = len(food)

print("Number of rows: ", n_before_cleaning)

Number of rows:  308585


This dataset is very large, and will require two parts for cleaning: null value handling and formatting. Addressing null values will inherently allow for a good amount of inspection, so we will start with this step. 

## Null Values

In [5]:
# Find the number of null values in each run
food.isna().sum()

Inspection ID          0
DBA Name               0
AKA Name            2422
License #             19
Facility Type       5323
Risk                  87
Address                0
City                 189
State                 67
Zip                   42
Inspection Date        0
Inspection Type        1
Results                0
Violations         86555
Latitude            1030
Longitude           1030
Location            1030
dtype: int64

There appears to be a considerable number of null values in this dataset. An alarming feature about these is that they are inconsisent between columns, which means null values in different columns will need to be treated individually. 


### Null values in 'AKA Name' Column

The first of these columns is 'AKA name'. Since there are no null values in 'DBA Name', we have a column consisting of names for each inspected location. This means the 'AKA name' columns is likely a secondary name, therefore not as crucial for our analysis. We can address these null values with duplicates of 'DBA Name'. To validate that this is an appriopriate method, we can compare the number of columns where 'DBA Name' and 'AKA Name' are the same and different. 

In [6]:
same_names = (food['DBA Name'] == food['AKA Name']).sum()
diff_names = (food['DBA Name'] != food['AKA Name']).sum()
print(f"The number of columns where 'DBA' and 'AKA' names are equal: {same_names}")
print(f"The number of columns where 'DBA' and 'AKA' names are not equal: {diff_names}")


The number of columns where 'DBA' and 'AKA' names are equal: 223922
The number of columns where 'DBA' and 'AKA' names are not equal: 84663


Since there are significantly more rows where 'DBA Name' and 'AKA Name' are equal, we will go ahead and replace the null values as described. 

In [7]:
food ['AKA Name'] = food['AKA Name'].fillna(food['DBA Name'])

# Check number of null values after cleaning
food.isna().sum()

Inspection ID          0
DBA Name               0
AKA Name               0
License #             19
Facility Type       5323
Risk                  87
Address                0
City                 189
State                 67
Zip                   42
Inspection Date        0
Inspection Type        1
Results                0
Violations         86555
Latitude            1030
Longitude           1030
Location            1030
dtype: int64

### Null values in 'License #' Column

The second column of interest, 'License #', is only missing 19. A small number of rows would be easy to delete from a dataset this large. Before going with that method, though, we will find if they represent something of interest. For example, a missing license number could represent a restaurant who received a violation for an expired license. To check this, we can isolate these 19 rows and find the violation that was reported. 

In [8]:
food['Violations'].loc[food['License #'].isna()].head(10)

5109                                                    NaN
43977     2. CITY OF CHICAGO FOOD SERVICE SANITATION CER...
74053     1. PERSON IN CHARGE PRESENT, DEMONSTRATES KNOW...
111833    2. CITY OF CHICAGO FOOD SERVICE SANITATION CER...
131849    2. CITY OF CHICAGO FOOD SERVICE SANITATION CER...
154076    30. FOOD IN ORIGINAL CONTAINER, PROPERLY LABEL...
174919    32. FOOD AND NON-FOOD CONTACT SURFACES PROPERL...
193333    21. * CERTIFIED FOOD MANAGER ON SITE WHEN POTE...
223184    34. FLOORS: CONSTRUCTED PER CODE, CLEANED, GOO...
226826                                                  NaN
Name: Violations, dtype: object

There does not seem to be any trend within the 'Violations' column, ruling out this cause. Before writing these rows off completely, we can rule out data collection errors by seeing if an existing license number exists in the data set for each restaurant with a null value. 

In [9]:
# Find the names of the restaurants with null 'License #' values
food['DBA Name'].loc[food['License #'].isna()]

5109                                  LEX CHINESE FOOD
43977                                     SUR LA TABLE
74053                                     SUR LA TABLE
111833                                    SUR LA TABLE
131849                                    SUR LA TABLE
154076                                    SUR LA TABLE
174919                                    SUR LA TABLE
193333                                    SUR LA TABLE
223184                                    SUR LA TABLE
226826                                 ARGENTINA FOODS
240506                                    SUR LA TABLE
241819                                    SUR LA TABLE
264717                        OLD ST. PATRICK'S CHURCH
268086                 GOD'S BATTLE AXE PRAYER ACADEMY
268299                 GOD'S BATTLE AXE PRAYER ACADEMY
269773                 GOD'S BATTLE AXE PRAYER ACADEMY
290147    AVALON COMMUNITY CHURCH/FREEDOM HOME ACADEMY
298362                             ST DEMETRIOS CHURCH
300944    

In [10]:
# Count rows with name 'Lex Chinese Food'
lex_chinese = (food['DBA Name'] == 'LEX CHINESE FOOD').sum()

# Count rows with name 'Sur La Table'
sur_latable = (food['DBA Name'] == 'SUR LA TABLE').sum()

# Count rows with name 'Argentina Foods'
argentina_foods = (food['DBA Name'] == 'ARGENTINA FOODS').sum()

# Count rows with name 'Old St. Patrick's Churck'
old_stp = (food['DBA Name'] == "OLD ST. PATRICK'S CHURCH").sum()

# Count rows with name 'Sur La Table'
gods_battleaxe = (food['DBA Name'] == "GOD'S BATTLE AXE PRAYER ACADEMY").sum()

# Count rows with name 'Avalon Community Church/Freedom Home Academy'
avalon = (food['DBA Name'] == 'AVALON COMMUNITY CHURCH/FREEDOM HOME ACADEMY').sum()

# Count rows with name 'St Demetrios Church'
st_demetrios = (food['DBA Name'] == 'ST DEMETRIOS CHURCH').sum()

# Print
print(f"Lex Chinese: {lex_chinese}")
print(f"Sur La Table: {sur_latable}")
print(f"Argentina Foods: {argentina_foods}")
print(f"Old St. Patrick's: {old_stp}")
print(f"God's Battle Axe: {gods_battleaxe}")
print(f"Avalong community chruch: {avalon}")
print(f"St. Demetrio's {st_demetrios}")

Lex Chinese: 13
Sur La Table: 11
Argentina Foods: 6
Old St. Patrick's: 9
God's Battle Axe: 3
Avalong community chruch: 1
St. Demetrio's 1


In [11]:
# For validation, print the 'License #' values for Lex Chinese Food rows
food['License #'].loc[food['DBA Name'] == 'LEX CHINESE FOOD']

4015      2738318.0
5109            NaN
16234     2738318.0
16264     2738318.0
29148     2738318.0
29608     2738318.0
46204     2738318.0
46594     2738318.0
66893     2738318.0
67351     2738318.0
83285     2738318.0
100248    2738318.0
100474    2738318.0
Name: License #, dtype: float64

In [12]:
# Fill null values based on existing License number values
food['License #'] = food['License #'].fillna(
    food.groupby('DBA Name')['License #'].transform('first')
)

# Print 'License #' values for Lex Chinese Food rows
food['License #'].loc[food['DBA Name'] == 'LEX CHINESE FOOD']


4015      2738318.0
5109      2738318.0
16234     2738318.0
16264     2738318.0
29148     2738318.0
29608     2738318.0
46204     2738318.0
46594     2738318.0
66893     2738318.0
67351     2738318.0
83285     2738318.0
100248    2738318.0
100474    2738318.0
Name: License #, dtype: float64

In [13]:
# Check number of null values after cleaning
food.isna().sum()

Inspection ID          0
DBA Name               0
AKA Name               0
License #              5
Facility Type       5323
Risk                  87
Address                0
City                 189
State                 67
Zip                   42
Inspection Date        0
Inspection Type        1
Results                0
Violations         86555
Latitude            1030
Longitude           1030
Location            1030
dtype: int64

Now that the number of null values in 'License #' is only 5, and since we filled the rows that we could, we will remove the remaining rows.

In [14]:
# Drop rows where 'License #' is null
food = food.dropna(subset = ['License #'])

# Check number of null values after cleaning
food.isna().sum()

Inspection ID          0
DBA Name               0
AKA Name               0
License #              0
Facility Type       5323
Risk                  87
Address                0
City                 189
State                 67
Zip                   42
Inspection Date        0
Inspection Type        1
Results                0
Violations         86553
Latitude            1030
Longitude           1030
Location            1030
dtype: int64

Now the number of null values in the 'License #' column is zero. 

The 'Facility Type' column can be crucial for our analysis. Also, there are over 5,000 missing values, which is a very significant amount. With these factors in mind, it is important that this column is handled carefully. First, we can get a sense of how many unique values are in this column with python. 

### Null Values in 'Facility Type' Column

In [15]:
# count unique values in 'Facility Type' column 
food['Facility Type'].nunique()

518

There are over 500 unique values in the 'Facility Type' column, meaning it would be too difficult to try to generalize null values into one of these unique names. This is likely due to formatting inconsistency, but that will be addressed later in the cleaning process. For now, we can take a look at a handful of rows where 'Facility Type' is null to see if we can find a trend.

In [16]:
# Print five rows where 'Facility Type' is null
food.loc[food['Facility Type'].isna() == True].head()

,Inspection ID,DBA Name,AKA Name,License #,Facility Type,Risk,Address,City,State,Zip,Inspection Date,Inspection Type,Results,Violations,Latitude,Longitude,Location
58,2633689,LUIGI'S PIZZA,LUIGI'S PIZZA,3078309.0,NaN,Risk 1 (High),344 N LARAMIE AVE,CHICAGO,IL,60644.0,04/08/2026,License,Not Ready,NaN,41.886995,-87.755467,"(41.886995469979524, -87.75546677076916)"
76,2633685,QUICK SHOP GROCERY,QUICK SHOP GROCERY,3077999.0,NaN,Risk 3 (Low),3255 W 63RD ST,CHICAGO,IL,60629.0,04/08/2026,License,Not Ready,NaN,41.778829,-87.705286,"(41.77882931533162, -87.70528574054822)"
90,2633726,PANDA EXPRESS #5046,PANDA EXPRESS,3078330.0,NaN,All,22 E CHICAGO AVE,CHICAGO,IL,60611.0,04/08/2026,License,Not Ready,NaN,41.896799,-87.627266,"(41.89679861905508, -87.62726628946932)"
149,2633620,LA CALACA,LA CALACA,3077860.0,NaN,Risk 1 (High),1800-1802 W PERSHING RD,CHICAGO,IL,60609.0,04/07/2026,License,Not Ready,NaN,41.823402,-87.670272,"(41.82340182276966, -87.67027172392757)"
177,2633530,TRADER JOE'S STORE #860,TRADER JOE'S,3078248.0,NaN,Risk 2 (Medium),6191 N LINCOLN AVE,CHICAGO,IL,60659.0,04/03/2026,License,Not Ready,NaN,41.994438,-87.713728,"(41.9944376008803, -87.71372779760488)"


Based on this output, something stands out immediately: two of these entries are corporations with multiple locations. This also shows us that when a company has multiple locations, the location number is listed in 'DBA Name', but a simplified version can be found in 'AKA Name'. Using this information, we can locate null values that share 'AKA Name' values with other observations, and assign 'Facility Type' values based on if one exists. First, we can create a new data frame consisting of only rows where 'Facility Type' is null, then count the number of unique 'AKA Name' values.

In [17]:
# Assign null_facility_types to the rows where 'Facility Type' is null
null_facility_types = food[food['Facility Type'].isna() == True]

# Count the number of unique 'AKA Name' values in null_facility_types
null_facility_types['AKA Name'].nunique()

4382

According to the python output, 4,382 out of the 5,323 null values in 'Facility Type' belong to a unique restaurant. This means that if another observation exists in our whole data frame where 'Facility Type' is not null, we can address almost 1,000 null values using this method. Now, we will make a list of the names of the unique 'AKA Name' values in our null_facility_types data frame, and search for the number of rows in our whole data frame consisting of these names.

In [18]:
# Create list of unique 'AKA Name' values in null_facility_types
nft_names = null_facility_types['AKA Name'].unique()

# Print first five entries for validation 
print(nft_names[0:5])

# Count the number of rows in the whole data frame that match our list of names
counter = 0
for name in food['AKA Name']:
    if name in nft_names:
        counter += 1

print(counter)

["LUIGI'S PIZZA" 'QUICK SHOP GROCERY' 'PANDA EXPRESS' 'LA CALACA'
 "TRADER JOE'S"]
31241


It looks like our names of interest are found over 31,000 times in the whole data frame, which is great news. From here, we can assign 'Facility Type' based on these other rows, like we did with 'License #'. 

In [19]:
# Fill null values based on existing 'Facility Type' values
food['Facility Type'] = food['Facility Type'].fillna(
    food.groupby('AKA Name')['Facility Type'].transform('first')
)

# Check number of null values after cleaning
food.isna().sum()

Inspection ID          0
DBA Name               0
AKA Name               0
License #              0
Facility Type       4359
Risk                  87
Address                0
City                 189
State                 67
Zip                   42
Inspection Date        0
Inspection Type        1
Results                0
Violations         86553
Latitude            1030
Longitude           1030
Location            1030
dtype: int64

In [20]:
# Again, assign null_facility_types to the rows where 'Facility Type' is null
null_facility_types = food[food['Facility Type'].isna() == True]

# Print first five rows to make sure companies with multiple locations were addressed
null_facility_types.head()

,Inspection ID,DBA Name,AKA Name,License #,Facility Type,Risk,Address,City,State,Zip,Inspection Date,Inspection Type,Results,Violations,Latitude,Longitude,Location
76,2633685,QUICK SHOP GROCERY,QUICK SHOP GROCERY,3077999.0,None,Risk 3 (Low),3255 W 63RD ST,CHICAGO,IL,60629.0,04/08/2026,License,Not Ready,NaN,41.778829,-87.705286,"(41.77882931533162, -87.70528574054822)"
149,2633620,LA CALACA,LA CALACA,3077860.0,None,Risk 1 (High),1800-1802 W PERSHING RD,CHICAGO,IL,60609.0,04/07/2026,License,Not Ready,NaN,41.823402,-87.670272,"(41.82340182276966, -87.67027172392757)"
202,2633636,SHANG NOODLE,SHANG NOODLE,3078084.0,None,Risk 1 (High),1613 N DAMEN AVE,CHICAGO,IL,60647.0,04/02/2026,License,Not Ready,NaN,41.910857,-87.677325,"(41.91085695465258, -87.67732537711214)"
226,2633635,SGD WICKER PARK,SGD WICKER PARK,3078170.0,None,Risk 1 (High),2411 W NORTH AVE,CHICAGO,IL,60647.0,04/02/2026,License,Not Ready,NaN,41.910205,-87.687702,"(41.910205129391386, -87.68770229368916)"
256,2633420,FORK AND COIN,FORK AND COIN,3078310.0,None,All,3938 N CENTRAL AVE,CHICAGO,IL,60634.0,04/01/2026,License,Not Ready,NaN,41.952525,-87.767127,"(41.952524733255466, -87.7671269338514)"


Using this method, we went from 5,323 null values to 4,359, addressing 964 null values at once. After reassigning the data frame containing rows where 'Facility Type' is null and printing the first five rows, we can also see that instances of companies with multiple locations have been addressed. This leaves us with locations that are likely all uniquely named, so grouping by name may not help us. However, we can broadly assign a 'Facility Type' to rows based on the contents of 'DBA Name'. For example, four rows from null_facility_types that are printed above show us five locations whose facility type can be deduced from their name. "Quick Shop Grocery" is a grocery store, "Shang Noodle" and "Fork and Coin" are restaurants, and "SGD Wicker Park" is a park. Using this information, we can construct lists of words that would be easily identify a location's facility type based on its name. 

Using the five rows above as a guide, we can test this method by creating a new column called 'Facility Type Test' to hold our newly attributed values. First, we will need to check the most common forms of formatting in the 'Facility Type' column. To do this, we can retreive the unique values from the column and look at the first 20. While filtering for unique values, we will also pass the .str.title() function, which automatically capitalizes the first letter of each word to avoid random formatting. 

In [21]:
# Address random formatting in 'Facility Type' column
food['Facility Type'] = food['Facility Type'].str.title()

# Find unique values from 'Facility Type'
facility_unique = food['Facility Type'].unique()

# Print the first ten values from the list of unique 'Facility Type' names
print(facility_unique[:20])


['Grocery Store' 'Mobile Food Preparer' 'Restaurant' 'School'
 "Children'S Services Facility" 'Daycare Combo 1586' None 'Catering'
 'Bakery' 'Long Term Care' 'Daycare Above And Under 2 Years'
 'Golden Diner' 'Shared Kitchen User (Long Term)' 'Liquor' 'Tavern'
 'Charter School' 'Daycare (Under 2 Years)' 'Bar' 'Shelter'
 'Convenience Store']


This output gives us a good example of how locations are categorized by facility type. It also shows us that facility type can get very specific, for example 'Tavern' and 'Bar' are listed as separate categories, even though being very similar in purpose. For the purpose of our analysis, we can choose to be more ambiguous with the way that we group facility types in favor of cleanliness. In order to maintain the original accuracy of the data set, we can keep grouped facility types and original facility types in separate columns. We will lose the ambiguity for rows that we do not have facility type data for, but this gives us the option of analyzing subsets of each broader facility type. 

We will define a list of words that would easily identify a location's facility type based on its name. Then, we will filter through locations with null values to check if their names contain these values, and assign a facility type based on its name. Since this method includes some ambiguity, we will test this method with a new, separate column.

In [22]:
# Create new column to hold duplicates of 'Facility Type' for testing
food['Facility Type Test'] = food['Facility Type']

# Create new column to hold duplicates of 'Facility Type' for grouping
food['Facility Type Grouped'] = food['Facility Type']

# Check that columns match 
food.isna().sum()

Inspection ID                0
DBA Name                     0
AKA Name                     0
License #                    0
Facility Type             4359
Risk                        87
Address                      0
City                       189
State                       67
Zip                         42
Inspection Date              0
Inspection Type              1
Results                      0
Violations               86553
Latitude                  1030
Longitude                 1030
Location                  1030
Facility Type Test        4359
Facility Type Grouped     4359
dtype: int64

#### *Children's Services Facility Type*

In [23]:
# Check 'DBA Name' for locations with "Children'S Services Facility"
print(food['DBA Name'].loc[food['Facility Type'] == "Children'S Services Facility"].head(10))

# Count the number of locations with this facility type that contain 'Academy' in the name
print(food['DBA Name'].loc[(food['DBA Name'].str.contains('ACADEMY')) & (food['Facility Type'] == "Children'S Services Facility")].count())

9                                       MONARCHS ACADEMY
24     CHERISH WITH LOVE AND CARE DAYCARE & LEARNING ...
44                             L&L ACADEMY AND PRESCHOOL
53                                MAYFAIR EARLY LEARNING
62                                          LA ESCUELITA
101             WONDERLAND CHILD CARE CENTER CORPORATION
117                           KIDS KINGDOM ACADEMY, INC.
144     HOUSE OF KIDDS CHILD CARE & LEARNING CENTER, INC
146                            LEE'S CUDDLES N CARE INC.
180                                     CATERPILLAR CARE
Name: DBA Name, dtype: object
1129


In [24]:
(food['Facility Type'].loc[food['DBA Name'].str.contains("ACADEMY")]).unique()

array(["Children'S Services Facility", 'School', 'Charter School',
       'Daycare Above And Under 2 Years', 'Daycare 2 Yrs To 12 Yrs',
       'Daycare (2 - 6 Years)', 'Daycare',
       "1023 Childern'S Services Facility", 'Daycare (Under 2 Years)',
       'Daycare Combo 1586', 'Restaurant', None, 'Public Shcool',
       'Private School', "1023-Children'S Services Facility",
       'Day Care 1023', '1584-Day Care Above 2 Years', 'Day Care'],
      dtype=object)

For the sake of our analysis, while converting null values, we can also convert rows with the above facility types to "Children'S Services Facility", since it is a more broad title. Before applying any changes to the original data, we will use the 'Facility Type Test' column to test this method. 

In [25]:
# Define list of filter words for location names
child_serv = ["DAYCARE", "PRESCHOOL", "EARLY LEARNING", "CHILD CARE", "KIDS", "ACADEMY", "CUDDLES", "INFANT", "TOT", "TOTS",
              "CHILDREN", "TODDLERS", 
              "DAY CARE", "PRE-SCHOOL", "DAY-CARE", "PRE SCHOOL", "CHILDCARE"]

# Define more specific list for 'Facility Type' conversion
child_serv_type = ["DAYCARE", "PRESCHOOL", "EARLY LEARNING", "CHILD CARE", "DAY CARE", "PRE-SCHOOL", "DAY-CARE", "PRE SCHOOL", "CHILDCARE", 
                   "1023-Children'S Services Facility", "1023 Children'S Services Facility"]

# Join list into a single string 
filter = "|".join(child_serv)

# Check for rows where 'Facility Type' is null that pass our filter 
apply_child_serv = (food['Facility Type'].isna()) & (food['DBA Name'].str.contains(filter, case = False, na = False))

# Join type list into a single string
type_filter = "|".join(child_serv_type)

# Check for rows where 'Facility Type' contains words in our child_serv_type filer
apply_child_serv_type = food['Facility Type'].str.contains(type_filter, case = False, na = False)

# Appy the changes to the test column and grouped column
food.loc[apply_child_serv, 'Facility Type Test'] = "Children'S Services Facility"
food.loc[apply_child_serv_type, 'Facility Type Grouped'] = "Children'S Services Facility"

# Check null values in data frame
food.isna().sum()


Inspection ID                0
DBA Name                     0
AKA Name                     0
License #                    0
Facility Type             4359
Risk                        87
Address                      0
City                       189
State                       67
Zip                         42
Inspection Date              0
Inspection Type              1
Results                      0
Violations               86553
Latitude                  1030
Longitude                 1030
Location                  1030
Facility Type Test        4323
Facility Type Grouped     4359
dtype: int64

In [26]:
# Check that our 'Facility Type' merge worked 
(food['Facility Type Test'].loc[food['DBA Name'].str.contains("ACADEMY")]).unique()

array(["Children'S Services Facility", 'School', 'Charter School',
       'Daycare Above And Under 2 Years', 'Daycare 2 Yrs To 12 Yrs',
       'Daycare (2 - 6 Years)', 'Daycare',
       "1023 Childern'S Services Facility", 'Daycare (Under 2 Years)',
       'Daycare Combo 1586', 'Restaurant', 'Public Shcool',
       'Private School', "1023-Children'S Services Facility",
       'Day Care 1023', '1584-Day Care Above 2 Years', 'Day Care'],
      dtype=object)

In [27]:
# Check rows with 'Childrens Services Facility' to check that make sure there were no obvious mismatches 
food['DBA Name'].loc[food['Facility Type Test'] == "Children'S Services Facility"].head(10)

9                                       MONARCHS ACADEMY
24     CHERISH WITH LOVE AND CARE DAYCARE & LEARNING ...
44                             L&L ACADEMY AND PRESCHOOL
53                                MAYFAIR EARLY LEARNING
62                                          LA ESCUELITA
101             WONDERLAND CHILD CARE CENTER CORPORATION
117                           KIDS KINGDOM ACADEMY, INC.
144     HOUSE OF KIDDS CHILD CARE & LEARNING CENTER, INC
146                            LEE'S CUDDLES N CARE INC.
180                                     CATERPILLAR CARE
Name: DBA Name, dtype: object

We were able to address 36 null values with this for this facility type alone, and we addressed some formatting that will benefit our analysis later on. After validating, we found that all 25/25 rows were obviously facilities for childrens' services, which is a good sign that we did not incorrectly label any locations. We will go ahead and apply these changes to the actual 'Facility Type' column. 

In [28]:
# Apply changes to 'Facility Type' and 'Facility Type Grouped' columns
food['Facility Type'] = food['Facility Type Test']
food['Facility Type Grouped'] = food['Facility Type Grouped'].fillna(food['Facility Type Test'])

# Remove unnessecary capitalized 'S' in "Children's Services Facility"
food.loc[food['Facility Type'] == "Children'S Services Facility", 'Facility Type'] = "Children's Services Facility"
food.loc[food['Facility Type Grouped'] == "Children'S Services Facility", 'Facility Type Grouped'] = "Children's Services Facility"

# Validate changes
food.isna().sum()

Inspection ID                0
DBA Name                     0
AKA Name                     0
License #                    0
Facility Type             4323
Risk                        87
Address                      0
City                       189
State                       67
Zip                         42
Inspection Date              0
Inspection Type              1
Results                      0
Violations               86553
Latitude                  1030
Longitude                 1030
Location                  1030
Facility Type Test        4323
Facility Type Grouped     4323
dtype: int64

In [29]:
# Count the number of rows with "Children's Services Facility" type
print("Rows grouped with 'Children's Services Facility' type: " + str(len(food.loc[food['Facility Type Grouped'] == "Children's Services Facility"])))

Rows grouped with 'Children's Services Facility' type: 16021


We can continue this approach for the other common types like 'Grocery Store' and 'Restaurant'.

#### *Grocery Store Type (Part one)*

In [30]:
# Repeat the process of checking 'DBA Name' for locations with "Grocery Store"
print(food['DBA Name'].loc[food['Facility Type'] == "Grocery Store"].head(10))

0           ONE STOP FOOD & LIQUOR STORE
19    Michoacana Supermarket Corporation
20           SPEEDYS FOOD AND SMOKE SHOP
26                 FAMILY GENERAL DOLLAR
37                      PETE'S MARKET #3
57                JEWEL FOOD STORE #3344
63                          DEVON MARKET
64                  SUPERMERCADO CHAPALA
69                      RYAN FOODS STORE
73                       ALI'S FOOD MART
Name: DBA Name, dtype: object


In [31]:
# Find the unique facility types that contain 'Market' in the name
(food['Facility Type'].loc[food['DBA Name'].str.contains("Market", case = False)]).unique()[:10]

array(['Grocery Store', 'Restaurant', 'Shared Kitchen User (Long Term)',
       None, 'Grocery/Taqueria', 'Catering', 'Grocery & Restaurant',
       'Liquor', 'Pop-Up Establishment Host-Tier Ii', 'Store'],
      dtype=object)

In [32]:
# Find the unique facility types that contain 'Mart' in the name
(food['Facility Type'].loc[food['DBA Name'].str.contains("Mart", case = False)]).unique()[:10]

array(['Grocery Store', 'Daycare Above And Under 2 Years', 'Restaurant',
       None, 'Grocery Store/Gas Station', "Children's Services Facility",
       'Liquor', 'Daycare (2 - 6 Years)', 'Restaurant/Grocery Store',
       'Shared Kitchen User (Long Term)'], dtype=object)

A significant subgroup of 'Grocery Store' has appeared twice: 'Convenience Store'. Because the distinction between these two stores is important yet hard to distinguish, we will address 'Convenience Store' first to avoid overwriting important information. 

#### *Convenience Store Type*

In [33]:
# Repeat the process of checking 'DBA Name' for locations with "Convenience Store" facility type
print((food['DBA Name'].loc[food['Facility Type'] == "Convenience Store"]).unique()[:10])

['7 ELEVEN #36033A' '7-ELEVEN #33496C' 'NEW YORK FOOD MART'
 'FRIENDLY FOOD MARKET' 'JORDAN GROCERY' 'Metromart Newsstand'
 'JAY CANDY STORE' '7-ELEVEN' 'J.J. PEPPERS' 'BLACKSEAMM']


In [34]:
# Check 'DBA Name' for locations with "Gas station" facility type
print((food['DBA Name'].loc[food['Facility Type'] == "Gas Station"]).unique()[:10])

['LOOMIS CITGO INC.' 'BP' 'CLARK-A & A GAS FUEL MINI FOOD MART'
 'ASHLAND WEST FALCON FUEL' 'CITGO' 'ROUX' 'MOBILE' 'AMSTAR' 'MARATHON'
 'GO LO']


In [35]:
# Find the unique facility types that contain 'Gas' in the name
(food['Facility Type'].loc[food['DBA Name'].str.contains("Gas", case = False)]).unique()

array(['Grocery Store', 'Gas Station Store', 'Restaurant', 'Gas Station',
       'Gas Station/Mini Mart', 'School', 'Grocery Store/Gas Station',
       None, 'Gas Station/Grocery', 'Convenience', 'Grocery/Gas Station',
       'Daycare Above And Under 2 Years',
       'Gas Station /Subway Mini Mart.'], dtype=object)

In [36]:
# Compare counts of facility types (grocery store vs convenience store) containing 'Mart' in the name 
print("Grocery stores with 'mart' in 'DBA Name': " + str(len(food.loc[(food['Facility Type'] == 'Grocery Store') & (food['DBA Name'].str.contains('MARKET'))])))
print("Convenience stores with 'mart' in 'DBA Name': " + str(len(food.loc[(food['Facility Type'].str.contains('Convenience', case = False)) & (food['DBA Name'].str.contains('MARKET'))])))

Grocery stores with 'mart' in 'DBA Name': 5728
Convenience stores with 'mart' in 'DBA Name': 12


These counts tell us that the word 'mart' is heaviliy associated with grocery stores, but we have many instances of convenience stores with 'mart' in the name. We can check the rows where 'Facility Type' is null and 'DBA Name' contains 'mart' to see if the names reveal any information about the locations. 

In [37]:
food['DBA Name'].loc[food['Facility Type'].isna() & food['DBA Name'].str.contains("Mart", case = False)].head(20)

396        BEST BUY MINI MART, INC
675            ROOT INN LUCKY MART
711                      79th MART
1895                    PRIME MART
2154           LUCKY FOOD MART LLC
9146            59TH MINI MART INC
14552                   DAILY MART
16236              MANNY FOOD MART
26485          MALAK FOOD MART INC
47291                TAYLOR'S MART
78276                TAYLOR'S MART
86970                BOB FOOD MART
87604          CALIFORNIA FOODMART
97136                  ELSTON MART
97464      AMOCO GAS AND FOOD MART
103696    SAMI CONVENIENT MART INC
104889        MORDI MINI MART INC.
124558            ZARIVA FOOD MART
128927    SAMI CONVENIENT MART INC
129853      MALIBU CONVENIENT MART
Name: DBA Name, dtype: object

This output reaffirms that the rows with a null value for 'Facility Type' with 'mart' in the name are clearly convenience stores, so we will continue with assigning 'Convenience Store' to these rows. We will also choose to combine similar facility types like gas station. 

In [38]:
# Define list of filter words for location names
convenience = ["MART", "MINIMART", "CONVENIENT", "CONVENIENCE", "GAS", "SNACKS", "SNACK", "CANDY", "BODEGA",
               "FUEL", "QUICK", "MINI FOOD MART", "NEWSSTAND", "NEWS STAND", "MINI FOODMART", "7-Eleven"]

# Define more specific list for 'Facility Type' conversion
convenience_type = ["Gas Station", "Mini Mart", "Convenience", "Subway"]

# Join list into a single string 
filter = "|".join(convenience)

# Check for rows where 'Facility Type' is null that pass our filter 
apply_convenience = (food['Facility Type'].isna()) & (food['DBA Name'].str.contains(filter, case = False, na = False))

# Join type list into a single string
type_filter = "|".join(convenience_type)

# Check for rows where 'Facility Type' contains words in our convenience_type filer
apply_convenience_type = food['Facility Type'].str.contains(type_filter, case = False, na = False)

# Appy the changes to the test column 
food.loc[apply_convenience, 'Facility Type Test'] = "Convenience Store"
food.loc[apply_convenience_type, 'Facility Type Grouped'] = "Convenience Store"

# Also, address instances of '7-Eleven' being recorded as 'Grocery Store'
food.loc[food['DBA Name'].str.contains("7-Eleven", case = False, na=False), 'Facility Type Test'] = 'Convenience Store'

# Check null values in data frame
food.isna().sum()

Inspection ID                0
DBA Name                     0
AKA Name                     0
License #                    0
Facility Type             4323
Risk                        87
Address                      0
City                       189
State                       67
Zip                         42
Inspection Date              0
Inspection Type              1
Results                      0
Violations               86553
Latitude                  1030
Longitude                 1030
Location                  1030
Facility Type Test        4046
Facility Type Grouped     4323
dtype: int64

In [39]:
# Check 25 rows where 'Facility Type Test' is 'Convenience Store'
food['DBA Name'].loc[food['Facility Type Test'] == 'Convenience Store'].head(25)

76           QUICK SHOP GROCERY
77                     7-Eleven
268           7-ELEVEN #37152 A
396     BEST BUY MINI MART, INC
410            7 ELEVEN #36033A
418      SUPERMERCADO LA BODEGA
562                    7-Eleven
675         ROOT INN LUCKY MART
711                   79th MART
1036          7-ELEVEN #36106 B
1249           7-ELEVEN #36718B
1895                 PRIME MART
2154        LUCKY FOOD MART LLC
3103           7-ELEVEN #13542L
3907           7-ELEVEN #33921B
4546           7-ELEVEN #33921B
4559                   7-ELEVEN
4834           7-ELEVEN #35978A
5047          7-ELEVEN # 32934E
5659            7-ELEVEN 29530D
5803          7-ELEVEN #26878 F
5918             7-ELEVEN 33609
6001                   7-ELEVEN
6309                   7-ELEVEN
6355      V'S CONVENIENCE STORE
Name: DBA Name, dtype: object

We were able to address almost 300 null values with this method, and we will apply the final changes to the actual 'Facility Type' column. 

In [40]:
# Apply changes to 'Facility Type' and 'Facility Type Groouped' columns
food['Facility Type'] = food['Facility Type Test']
food['Facility Type Grouped'] = food['Facility Type Grouped'].fillna(food['Facility Type Test'])

# Validate changes
food.isna().sum()

Inspection ID                0
DBA Name                     0
AKA Name                     0
License #                    0
Facility Type             4046
Risk                        87
Address                      0
City                       189
State                       67
Zip                         42
Inspection Date              0
Inspection Type              1
Results                      0
Violations               86553
Latitude                  1030
Longitude                 1030
Location                  1030
Facility Type Test        4046
Facility Type Grouped     4046
dtype: int64

In [41]:
# Count the number of rows with 'Convenience Store' type
print("Rows grouped with 'Convenience Store' type: " + str(len(food.loc[food['Facility Type Grouped'] == "Convenience Store"])))

Rows grouped with 'Convenience Store' type: 840


#### *Grocery Store Type (continued)*

Now that instances of 'Convenience Store' have been addressed, we can perform the same methods for finding filter words in the 'DBA Name' column.

In [42]:
 # Repeat the process of checking 'DBA Name' for locations with "Grocery Store"
print(food['DBA Name'].loc[food['Facility Type'] == "Grocery Store"].head(20))

0            ONE STOP FOOD & LIQUOR STORE
19     Michoacana Supermarket Corporation
20            SPEEDYS FOOD AND SMOKE SHOP
26                  FAMILY GENERAL DOLLAR
37                       PETE'S MARKET #3
57                 JEWEL FOOD STORE #3344
63                           DEVON MARKET
64                   SUPERMERCADO CHAPALA
69                       RYAN FOODS STORE
73                        ALI'S FOOD MART
98                            VARSHA FOOD
103                       LINDO MICHOACAN
112                  FAMILY DOLLAR # 7726
121                 CARNICERIA JIMENEZ #2
122                      LEAMINGTON FOODS
164                        PAXTON GROCERY
168                      O & M QUICK MART
175                      7 DAYS FOOD MART
177               TRADER JOE'S STORE #860
179                      HABEEB MEAT STOP
Name: DBA Name, dtype: object


In [43]:
# Check facility types that contain 'Food' in the 'DBA Name' column 
(food['Facility Type'].loc[food['DBA Name'].str.contains("Food", case = False)]).unique()[:10]


array(['Grocery Store', 'Mobile Food Preparer', 'Restaurant',
       'Charter School', None, 'Private School', 'Convenience Store',
       'Liquor', 'Pop-Up Food Establishment User-Tier Ii', 'School'],
      dtype=object)

In [44]:
# Check facility types that contain 'Market' in the 'DBA Name' column 
(food['Facility Type'].loc[food['DBA Name'].str.contains("Market", case = False)]).unique()[:20]

array(['Grocery Store', 'Restaurant', 'Shared Kitchen User (Long Term)',
       None, 'Grocery/Taqueria', 'Catering', 'Grocery & Restaurant',
       'Liquor', 'Pop-Up Establishment Host-Tier Ii', 'Store',
       'Grocery/Deli', 'Commissary', "Children'S Services Facility",
       'Convenience Store', 'French Market Space', 'Shared Kitchen',
       'Grocery Store/Bakery', 'Bakery', 'Mobile Food Preparer',
       'Wholesale'], dtype=object)

In [45]:
# Check 'DBA Name' for rows with 'Store' facility type
print(food['DBA Name'].loc[food['Facility Type'] == "Store"].head(20))

31947                         IT'SUGAR
32147            COMPLETELY NUTS, INC.
60666     DIVERSEY DOLLAR ISLAND, INC.
61914                      AZ FOODMART
63606                         IT'SUGAR
64190                GATEWAY NEWSTANDS
65121            COMPLETELY NUTS, INC.
66468             D & D WINE & SPIRITS
66976                B & B SUPERMARKET
80527          DIVISION PETROLEUM INC.
80984             D & D WINE & SPIRITS
80998             D & D WINE & SPIRITS
83069                      COFFEE CURE
86416                  NADIA FOOD MART
87202                   HOMAN MART INC
100640    DIVERSEY DOLLAR ISLAND, INC.
100848    DIVERSEY DOLLAR ISLAND, INC.
101200    DIVERSEY DOLLAR ISLAND, INC.
105564                        IT'SUGAR
110589                     AZ FOODMART
Name: DBA Name, dtype: object


In [46]:
# Define list of filter words for location names
grocery = ["MARKET", "GROCER", "GROCERY", "GROCERS", "DELI",
           "SUPER MARKET", "SUPERMARKET", "MERCADO", "SUPERMERCADO", "FAMILY DOLLAR", "SAVE", "SAVER"]

# Define more specific list for 'Facility Type' conversion
grocery_type = ["Grocery", "Supermarket", "Market", "Grocer", "Deli", "Wholesale"]

# Join list into a single string 
filter = "|".join(grocery)

# Check for rows where 'Facility Type' is null that pass our filter 
apply_grocery = (food['Facility Type'].isna()) & (food['DBA Name'].str.contains(filter, case = False, na = False))

# Join type list into a single string
type_filter = "|".join(grocery_type)

# Check for rows where 'Facility Type' contains words in our type_filter filer
apply_grocery_type = food['Facility Type'].str.contains(type_filter, case = False, na = False)

# Join locations with 'Store' facility type with 'Grocery Store'
food.loc[food['Facility Type'] == 'Store', 'Facility Type Test'] = 'Grocery Store'

# Appy the changes to the test column 
food.loc[apply_grocery, 'Facility Type Test'] = "Grocery Store"
food.loc[apply_grocery_type, 'Facility Type Grouped'] = "Grocery Store"

# Check null values in data frame
food.isna().sum()

Inspection ID                0
DBA Name                     0
AKA Name                     0
License #                    0
Facility Type             4046
Risk                        87
Address                      0
City                       189
State                       67
Zip                         42
Inspection Date              0
Inspection Type              1
Results                      0
Violations               86553
Latitude                  1030
Longitude                 1030
Location                  1030
Facility Type Test        3703
Facility Type Grouped     4046
dtype: int64

In [47]:
# Check rows where 'Facility Type Test' is equal to 'Grocery Store'
food['DBA Name'].loc[food['Facility Type Test'] == 'Grocery Store'].head(10)

0           ONE STOP FOOD & LIQUOR STORE
19    Michoacana Supermarket Corporation
20           SPEEDYS FOOD AND SMOKE SHOP
26                 FAMILY GENERAL DOLLAR
37                      PETE'S MARKET #3
57                JEWEL FOOD STORE #3344
63                          DEVON MARKET
64                  SUPERMERCADO CHAPALA
69                      RYAN FOODS STORE
73                       ALI'S FOOD MART
Name: DBA Name, dtype: object

We were able to address almost 400 instances of grocery store with this method, and we will apply to the true 'Facility Type' column. 

In [48]:
# Apply changes to 'Facility Type' and 'Facility Type Grouped' columns
food['Facility Type'] = food['Facility Type Test']
food['Facility Type Grouped'] = food['Facility Type Grouped'].fillna(food['Facility Type Test'])

# Validate changes
food.isna().sum()

Inspection ID                0
DBA Name                     0
AKA Name                     0
License #                    0
Facility Type             3703
Risk                        87
Address                      0
City                       189
State                       67
Zip                         42
Inspection Date              0
Inspection Type              1
Results                      0
Violations               86553
Latitude                  1030
Longitude                 1030
Location                  1030
Facility Type Test        3703
Facility Type Grouped     3703
dtype: int64

In [49]:
# Count rows grouped with 'Grocery Store' type
print("Rows grouped with 'Grocery Store' type: " + str(len(food.loc[food['Facility Type Grouped'] == "Grocery Store"])))

Rows grouped with 'Grocery Store' type: 38842


#### *Restaurant Type*

Before starting the 'Restaurant' facility type, we will check some of the remaining null values in the 'Facility Type' column, as well as the current number of unique 'Facility Type' values in our dataset. 

In [50]:
# Count null values in 'Facility Type' 
food.loc[food['Facility Type'].isna()].head(10)

,Inspection ID,DBA Name,AKA Name,License #,Facility Type,Risk,Address,City,State,Zip,Inspection Date,Inspection Type,Results,Violations,Latitude,Longitude,Location,Facility Type Test,Facility Type Grouped
149,2633620,LA CALACA,LA CALACA,3077860.0,None,Risk 1 (High),1800-1802 W PERSHING RD,CHICAGO,IL,60609.0,04/07/2026,License,Not Ready,NaN,41.823402,-87.670272,"(41.82340182276966, -87.67027172392757)",None,None
202,2633636,SHANG NOODLE,SHANG NOODLE,3078084.0,None,Risk 1 (High),1613 N DAMEN AVE,CHICAGO,IL,60647.0,04/02/2026,License,Not Ready,NaN,41.910857,-87.677325,"(41.91085695465258, -87.67732537711214)",None,None
226,2633635,SGD WICKER PARK,SGD WICKER PARK,3078170.0,None,Risk 1 (High),2411 W NORTH AVE,CHICAGO,IL,60647.0,04/02/2026,License,Not Ready,NaN,41.910205,-87.687702,"(41.910205129391386, -87.68770229368916)",None,None
256,2633420,FORK AND COIN,FORK AND COIN,3078310.0,None,All,3938 N CENTRAL AVE,CHICAGO,IL,60634.0,04/01/2026,License,Not Ready,NaN,41.952525,-87.767127,"(41.952524733255466, -87.7671269338514)",None,None
291,2633293,LA COCINA DE CARMELITA,LA COCINA DE CARMELITA,2966330.0,None,Risk 2 (Medium),2421 S SPAULDING AVE HS,CHICAGO,IL,60623.0,03/31/2026,Canvass,Pass,NaN,41.847459,-87.707416,"(41.84745866023468, -87.70741581960564)",None,None
298,2633359,SOLI ON THE FLY,SOLI ON THE FLY,3074133.0,None,All,10000 W OHARE ST,CHICAGO,IL,60666.0,03/31/2026,License,Not Ready,NaN,41.976201,-87.905309,"(41.97620113902387, -87.90530912510346)",None,None
415,2633218,WESTERN FOOD & TOBACCO,WESTERN FOOD & TOBACCO,3073319.0,None,Risk 2 (Medium),6257 S WESTERN AVE,CHICAGO,IL,60636.0,03/27/2026,License,Not Ready,NaN,41.779367,-87.683497,"(41.7793672500998, -87.68349741716884)",None,None
564,2633024,PAKS ON THE PIER,PAKS ON THE PIER,3077934.0,None,All,600 E GRAND AVE,CHICAGO,IL,60611.0,03/24/2026,License,No Entry,NaN,41.892094,-87.611570,"(41.892094136861786, -87.61156988394656)",None,None
725,2632858,GABY'S FUNNEL CAKES #2 INC,GABY'S FUNNEL CAKES #2 INC,2956841.0,None,Risk 2 (Medium),3814 S HOME AVE,BERWYN,IL,60402.0,03/20/2026,Canvass,Pass,NaN,NaN,NaN,NaN,None,None
748,2632787,HOKKAIDO RAMEN SANTOUKA,HOKKAIDO RAMEN SANTOUKA,3074028.0,None,Risk 1 (High),160 N HALSTED ST,CHICAGO,IL,60661.0,03/19/2026,License,Not Ready,NaN,41.884873,-87.647596,"(41.88487322678193, -87.64759638955412)",None,None


In [51]:
# Find the number of unique 'Facility Type' values
print(len(food['Facility Type'].unique()))

466


We have slimmed down the number of unique facility types from over 500 to 342, which will greatly benefit our analysis. We can look at some of these rows to get a feeling of what facility types are left to be cleaned. 

In [52]:
# Print the unique facility type values 
food['Facility Type'].unique()[:10]

array(['Grocery Store', 'Mobile Food Preparer', 'Restaurant', 'School',
       "Children'S Services Facility", 'Daycare Combo 1586',
       'Convenience Store', 'Catering', None, 'Bakery'], dtype=object)

This output actually shows us good news: there are still a lot of recorded 'Facility Type' values that can be merged with 'restaurant' ('Golden Diner', 'The Infused Bakery', 'Shared Kitchen'). This means that the 'restaurant' type will likely require more cleaning than the other sections, but it also the potential to address more irregularity than the prior types. 

In [53]:
# Repeat the process of checking 'DBA Name' for locations with "Restaurant" facility type
print((food['DBA Name'].loc[food['Facility Type'] == "Restaurant"]).unique()[:10])

['THE BBQ SHACK' "TONY'S TACOS AND BURGERS,INC." 'SHARKS FISH  &  CHICKEN'
 'LEVEL SPORTING CLUB' 'HALAL FOOD NYC' 'KARACHI CHAAT HOUSE 1'
 'MANCHAMANTELES' 'CASA CAFE' "SAZON ZULIANO PA' QUE EL OSO, LLC"
 'EL POLLO CRIS CRIS 2']


In [54]:
# Find names that correspond with 'Golden Diner'
print(food['DBA Name'].loc[food['Facility Type'] == "Golden Diner"].head(10))

190              SOUTHEAST CENTER (ATLAS)
412                   Central/West Center
463              LINCOLN PERRY APARTMENTS
2671                  HILLIARD APARTMENTS
2747              Britton Budd Apartments
3089             CHINESE COMMUNITY CENTER
3440          Patrick Sullivan Apartments
3975          Patrick Sullivan Apartments
6690    PRINCETON APARTMENTS GOLDEN DINER
7152              LAS AMERICAS APARTMENTS
Name: DBA Name, dtype: object


In [55]:
# Find names that correspond with 'Shared Kitchen'
print(food['DBA Name'].loc[food['Facility Type'] == "Shared Kitchen"].head(10))

924                               NIMBUS-5 LLC
936     CHICAGO TEACHERS UNION FOUNDATION, INC
1613                      Kitchen Chicago, LLC
1695                              THE HATCHERY
1937                                 CRAVE B2B
2005    CHICAGO TEACHERS UNION FOUNDATION, INC
2089                              THE HATCHERY
2338             SHAVED ICE DISTRIBUTORS, INC.
4540                             CLOUDKITCHENS
4691                  PREMIER RESTAURANT GROUP
Name: DBA Name, dtype: object


Restaurants appear to have much more variability in their 'DBA Name' column. For instance, locations with the 'Golden Diner' facility have names that correspond with apartment buildings, since cafeterias are categorized under this type. However, the distinction between a dining hall and restaurant is not significant for our analysis, so to address these values we will have to filter by the facility type itself. 

In [56]:
# Define list of filter words for location names
restaurant = ["BBQ", "SHACK", "CHICKEN", "FRIED CHICKEN", "FOOD", "GRILL", "MEDITERRANEAN", "COFFEE", "CAFE", "NOODLE", "NOODLES", "BAO",
              "BISTRO", "TACOS", "BURGERS", "BURGER", "TAQUERIA", "TORTILLERIA", "POLLO", "SUSHI", "STEAK", "STEAKHOUSE", "LOUNGE", "BANQUET", "HALL",
              "BAR", "TAVERN", "RESTAURANT", "EATERY", "DRIVEIN", "DRIVE IN", "DRIVE-IN", "BAGELS", "DONUTS", "TACO",
              "CHINESE", "MEXICAN", "MIDDLE-EASTERN", "MIDDLE EASTERN", "HIBACHI", "KITCHEN", "CATERING"]

# Define more specific list for 'Facility Type' conversion
restaurant_type = ["Restaurant/", "Catering", "Golden Diner", "Tavern", "Bar", "Cafe", "Bistro", "Kitchen", "Food Vendor",
                   "Mobile Food", "Food Truck", "Bakery", "Coffee", "Lounge", "Hall", "Banquet"]

# Join list into a single string 
filter = "|".join(restaurant)

# Check for rows where 'Facility Type' is null that pass our filter 
apply_restaurant = (food['Facility Type'].isna()) & (food['DBA Name'].str.contains(filter, case = False, na = False))

# Join type list into a single string
type_filter = "|".join(restaurant_type)

# Check for rows where 'Facility Type' contains words in our type_filter filer
apply_restaurant_type = food['Facility Type'].str.contains(type_filter, case = False, na = False)


# Appy the changes to the test column 
food.loc[apply_restaurant, 'Facility Type Test'] = "Restaurant"
food.loc[apply_restaurant_type, 'Facility Type Test'] = "Restaurant"

# Check null values in data frame
food.isna().sum()

Inspection ID                0
DBA Name                     0
AKA Name                     0
License #                    0
Facility Type             3703
Risk                        87
Address                      0
City                       189
State                       67
Zip                         42
Inspection Date              0
Inspection Type              1
Results                      0
Violations               86553
Latitude                  1030
Longitude                 1030
Location                  1030
Facility Type Test        2285
Facility Type Grouped     3703
dtype: int64

In [57]:
# Check rows where 'Facility Type Test' is equal to 'Restaurant'
food['DBA Name'].loc[food['Facility Type Test'] == 'Restaurant'].head(10)

1         MR QUILES MEXICAN FOOD #2
2          LA GUERITA MEXICAN SHACK
3                     THE BBQ SHACK
5     TONY'S TACOS AND BURGERS,INC.
6            MR QUILES MEXICAN FOOD
7           SHARKS FISH  &  CHICKEN
8               LEVEL SPORTING CLUB
10                   HALAL FOOD NYC
11            KARACHI CHAAT HOUSE 1
13                   MANCHAMANTELES
Name: DBA Name, dtype: object

As anticipated, this method addressed the highest number of null values at almost 1,300. We will apply these changes to the true 'Facility Type' column and count the number of values in each for each of the 'Facility Type' values that we have addressed to validate that no information is being overwritten. 

In [58]:
# Apply changes to 'Facility Type' and 'Facility Type Grouped' columns
food['Facility Type'] = food['Facility Type Test']
food['Facility Type Grouped'] = food['Facility Type Grouped'].fillna(food['Facility Type Test'])

# Validate changes
food.isna().sum()

Inspection ID                0
DBA Name                     0
AKA Name                     0
License #                    0
Facility Type             2285
Risk                        87
Address                      0
City                       189
State                       67
Zip                         42
Inspection Date              0
Inspection Type              1
Results                      0
Violations               86553
Latitude                  1030
Longitude                 1030
Location                  1030
Facility Type Test        2285
Facility Type Grouped     2285
dtype: int64

In [59]:
# Check counts of each facility type 
print("Rows grouped with 'Childrens Services Facility' type: " + str(len(food.loc[food['Facility Type Grouped'] == "Children's Services Facility"])))
print("Rows grouped with 'Convenience Store' type: " + str(len(food.loc[food['Facility Type Grouped'] == "Convenience Store"])))
print("Rows grouped with 'Grocery Store' type: " + str(len(food.loc[food['Facility Type Grouped'] == "Grocery Store"])))
print("Rows grouped with 'Restaurant' type: " + str(len(food.loc[food['Facility Type Grouped'] == "Restaurant"])))

Rows grouped with 'Childrens Services Facility' type: 16021
Rows grouped with 'Convenience Store' type: 721
Rows grouped with 'Grocery Store' type: 38842
Rows grouped with 'Restaurant' type: 211292


We will also check the number of unique values in 'Facility Type' to quantify the number of facility types that were grouped together. 

In [60]:
# Find the number of unique 'Facility Type' values
print(len(food['Facility Type Grouped'].unique()))

372


To ensure that we addressed as many instances of 'Restaurant' as possible, we can read some of the remaining null values in 'Facility Type' to see if any should obviously grouped together with the other restaurants. 

In [61]:
# Read the names of 10 rows with null 'Facility Type' values
food['DBA Name'].loc[food['Facility Type'].isna()].head(10)

149                     LA CALACA
226               SGD WICKER PARK
256                 FORK AND COIN
291        LA COCINA DE CARMELITA
298               SOLI ON THE FLY
564              PAKS ON THE PIER
725    GABY'S FUNNEL CAKES #2 INC
748       HOKKAIDO RAMEN SANTOUKA
866        MERCADITO LINCOLN PARK
956                         JOREE
Name: DBA Name, dtype: object

After running this query, there are some obvious stragglers that slipped through our first filter. For example, 'Kitchen' was included in our filter, but 'Cocina' was not. We will create a new filter list and pass the adjustment to the dataset again. 

In [62]:
# Create list of filter words
restaurant = ["COCINA", "RAMEN", "SAIGON", "FORK", "PAKS", "CAKES", "CAKE", "PASTRY", "PASTRIES", "SABRI", "GOLFSTROMMEN",
              "JERK", "PHO", "BANH MI", "MERCADITO"] # For some words like Sabri, Golfstrommen, and Mercadito, Google was referred to because of their uniqueness


# Join list into a single string 
filter = "|".join(restaurant)

# Check for rows where 'Facility Type' is null that pass our filter 
apply_restaurant = (food['Facility Type'].isna()) & (food['DBA Name'].str.contains(filter, case = False, na = False))

# Appy the changes to the test column 
food.loc[apply_restaurant, 'Facility Type Test'] = "Restaurant"

# Check null values in data frame
food.isna().sum()

Inspection ID                0
DBA Name                     0
AKA Name                     0
License #                    0
Facility Type             2285
Risk                        87
Address                      0
City                       189
State                       67
Zip                         42
Inspection Date              0
Inspection Type              1
Results                      0
Violations               86553
Latitude                  1030
Longitude                 1030
Location                  1030
Facility Type Test        2245
Facility Type Grouped     2285
dtype: int64

In [63]:
# Check rows where 'Facility Type Test' is equal to 'Restaurant'
food['DBA Name'].loc[food['Facility Type Test'] == 'Restaurant'].head(10)

1         MR QUILES MEXICAN FOOD #2
2          LA GUERITA MEXICAN SHACK
3                     THE BBQ SHACK
5     TONY'S TACOS AND BURGERS,INC.
6            MR QUILES MEXICAN FOOD
7           SHARKS FISH  &  CHICKEN
8               LEVEL SPORTING CLUB
10                   HALAL FOOD NYC
11            KARACHI CHAAT HOUSE 1
13                   MANCHAMANTELES
Name: DBA Name, dtype: object

We caught 40 more rows that we didn't have before. We can apply these changes and check one more time. 

In [64]:
# Apply changes to 'Facility Type' and 'Facility Type Grouped' columns
food['Facility Type'] = food['Facility Type Test']
food['Facility Type Grouped'] = food['Facility Type Grouped'].fillna(food['Facility Type Test'])

# Validate changes
food.isna().sum()

Inspection ID                0
DBA Name                     0
AKA Name                     0
License #                    0
Facility Type             2245
Risk                        87
Address                      0
City                       189
State                       67
Zip                         42
Inspection Date              0
Inspection Type              1
Results                      0
Violations               86553
Latitude                  1030
Longitude                 1030
Location                  1030
Facility Type Test        2245
Facility Type Grouped     2245
dtype: int64

In [65]:
# Read the names of 10 rows with null 'Facility Type' values
food['DBA Name'].loc[food['Facility Type'].isna()].head(10)


149                                LA CALACA
226                          SGD WICKER PARK
298                          SOLI ON THE FLY
956                                    JOREE
1185    DEPARTMENT OF REHABILITATION SVC/VFP
1349                     MOLINO LOS HERMANOS
1512                               TARRA LLC
1899                            FARMERS MILK
2122    DEPARTMENT OF REHABILITATION SVC/VFP
2288                           LARAMIE SHELL
Name: DBA Name, dtype: object

In [66]:
print("Rows grouped with 'Restaurant' type: " + str(len(food.loc[food['Facility Type Grouped'] == "Restaurant"])))

Rows grouped with 'Restaurant' type: 211332


Judging from these 25 names, it is not obvious that any are restaurants. We will move on from validating 'Restaurant' types, because 211,234 rows is enough for an analysis. 

#### *Pharmacy Facility Type*

Something that stands out about the output from the last query of isna() on the 'Facility Type' column is that there are multiple instances of CVS Pharmacy. This is an obvious instand of a 'Pharmacy' facility type that was left unrecorded. We can look repeat the earlier process of creating filter words for these locations. 

In [67]:
# Repeat the process of checking 'DBA Name' for locations with "Pharmacy" facility type
print((food['DBA Name'].loc[food['Facility Type'] == "Pharmacy"]).unique()[:10])

['RAVENSWOOD PHARMACY' 'WALGREENS #13973']


In [68]:
# Find facility types that are associated with DBA Names consisting of 'Pharmacy'
(food['Facility Type'].loc[food['DBA Name'].str.contains("Pharmacy", na = False, case = False)]).unique()[:10]

array([None, 'Grocery Store', 'Convenience', 'Convenience/Drug Store',
       'Pharmacy', 'Grocery Store /Pharmacy', 'Liquor'], dtype=object)

In [69]:
# Check 'DBA Name' for locations with "Drug Store" in facility type
print((food['DBA Name'].loc[food['Facility Type'].str.contains("Drug Store", na = False, case = False)]).unique()[:10])

['WALGREENS # 13106' 'CVS/PHARMACY #4061' 'WALGREENS #21160'
 'WALGREENS #05192' 'WALGREENS BOND DRUG COMPANY OF']


In [70]:
# Define list of filter words for location names
pharmacy = ["PHARMACY", "DRUG STORE", "CVS", "DRUG COMPANY", "WALGREENS"]

# Define more specific list for 'Facility Type' conversion
pharmacy_type = ["Pharmacy", "Drug Store", "Pharmacist"]

# Join list into a single string 
filter = "|".join(pharmacy)

# Check for rows where 'Facility Type' is null that pass our filter 
apply_pharmacy = (food['Facility Type'].isna()) & (food['DBA Name'].str.contains(filter, case = False, na = False))

# Join type list into a single string
type_filter = "|".join(pharmacy_type)

# Check for rows where 'Facility Type' contains words in our type_filter filer
apply_pharmacy_type = food['Facility Type'].str.contains(type_filter, case = False, na = False)


# Appy the changes to the test column 
food.loc[apply_pharmacy, 'Facility Type Test'] = "Pharmacy"
food.loc[apply_pharmacy_type, 'Facility Type Grouped'] = "Pharmacy"

# Check null values in data frame
food.isna().sum()

Inspection ID                0
DBA Name                     0
AKA Name                     0
License #                    0
Facility Type             2245
Risk                        87
Address                      0
City                       189
State                       67
Zip                         42
Inspection Date              0
Inspection Type              1
Results                      0
Violations               86553
Latitude                  1030
Longitude                 1030
Location                  1030
Facility Type Test        2228
Facility Type Grouped     2245
dtype: int64

In [71]:
# Apply changes to 'Facility Type' and 'Facility Type Grouped' columns
food['Facility Type'] = food['Facility Type Test']
food['Facility Type Grouped'] = food['Facility Type Grouped'].fillna(food['Facility Type Test'])

# Validate changes
food.isna().sum()

Inspection ID                0
DBA Name                     0
AKA Name                     0
License #                    0
Facility Type             2228
Risk                        87
Address                      0
City                       189
State                       67
Zip                         42
Inspection Date              0
Inspection Type              1
Results                      0
Violations               86553
Latitude                  1030
Longitude                 1030
Location                  1030
Facility Type Test        2228
Facility Type Grouped     2228
dtype: int64

In [72]:
# Check number of rows grouped with 'Pharmacy' type
print("Rows grouped with 'Pharmacy' type: " + str(len(food.loc[food['Facility Type Grouped'] == "Pharmacy"])))

Rows grouped with 'Pharmacy' type: 53


Now, we will check the total number of rows that are grouped by each facility type we have cleaned. We can use these numbers to get a proportion of the total dataset, and assess if further cleaning is necessary for this column.

In [73]:
# Get number of rows grouped with "Children's Services Facility" type
n_csf = int(len(food.loc[food['Facility Type Grouped'] == "Children's Services Facility"]))

# Get number of rows grouped with 'Convenience Store' type
n_convenience = int(len(food.loc[food['Facility Type Grouped'] == "Convenience Store"]))

# Get number of rows grouped with 'Grocery Store' type
n_grocery = int(len(food.loc[food['Facility Type Grouped'] == "Grocery Store"]))

# Get number of rows grouped with "Restaurant" type
n_restaurant = int(len(food.loc[food['Facility Type Grouped'] == "Restaurant"]))

# Get number of rows grouped with 'Pharmacy' type
n_pharmacy = int(len(food.loc[food['Facility Type Grouped'] == "Pharmacy"]))

# Get total number of rows in 'food'
n = len(food)


print("Rows grouped with 'Childrens Services Facility' type: " + str(n_csf))
print("Rows grouped with 'Convenience Store' type: " + str(n_convenience))
print("Rows grouped with 'Grocery Store' type: " + str(n_grocery))
print("Rows grouped with 'Restaurant' type: " + str(n_restaurant))
print("Rows grouped with 'Pharmacy' type: " + str(n_pharmacy))
print(f"Total cleaned rows: {n_csf + n_convenience + n_grocery + n_restaurant + n_pharmacy}")
print("Total rows: " + str(n))
print(f"Difference: {n - (n_csf + n_convenience + n_grocery + n_restaurant + n_pharmacy) }")

Rows grouped with 'Childrens Services Facility' type: 16021
Rows grouped with 'Convenience Store' type: 710
Rows grouped with 'Grocery Store' type: 38828
Rows grouped with 'Restaurant' type: 211332
Rows grouped with 'Pharmacy' type: 53
Total cleaned rows: 266944
Total rows: 308580
Difference: 41636


In [74]:
# Get the proportion of rows cleaned out of total rows 
print(f"Proportion of cleaned rows: {round((n_csf + n_convenience + n_grocery + n_restaurant + n_pharmacy) / n, 2)}")



Proportion of cleaned rows: 0.87


We have meaningfully addresssed 87% of the overall data in the entire dataset, leaving us with 266,944 rows of clean, actionable data. Since there are still 41,000 rows, these may be insighful during mapping or other analysis. Instead of directly dropping these rows, we can assign 'Unknown' to the 'Facility Type' column where they are null. 

In [75]:
# Fill null values in the 'Facility Type' and 'Facility Type Grouped' columns with a string containing 'Unknown'
food['Facility Type'] = food['Facility Type'].fillna('Unknown')
food['Facility Type Grouped'] = food['Facility Type Grouped'].fillna('Unknown')

# Also, drop 'Facilty Type Test' column 
food = food.drop(columns='Facility Type Test')

# Validate that cleaning was successful
food.isna().sum()

Inspection ID                0
DBA Name                     0
AKA Name                     0
License #                    0
Facility Type                0
Risk                        87
Address                      0
City                       189
State                       67
Zip                         42
Inspection Date              0
Inspection Type              1
Results                      0
Violations               86553
Latitude                  1030
Longitude                 1030
Location                  1030
Facility Type Grouped        0
dtype: int64

#### 'Risk' Column

First, we will simply observe the first 10 rows where 'Risk' is null to see if there are any obvious trends. 

In [76]:
# Print first 10 rows where 'Risk' is null
food.loc[food['Risk'].isna()].head(10)

,Inspection ID,DBA Name,AKA Name,License #,Facility Type,Risk,Address,City,State,Zip,Inspection Date,Inspection Type,Results,Violations,Latitude,Longitude,Location,Facility Type Grouped
1097,2632456,BAD BUTTER,BAD BUTTER,3065320.0,Restaurant,NaN,1653-1655 W CORTLAND ST,CHICAGO,IL,60622.0,03/12/2026,License,Not Ready,NaN,41.915969,-87.669959,"(41.91596940930692, -87.66995937984925)",Restaurant
1536,2631987,HUBBARD INN,HUBBARD INN,3069545.0,Restaurant,NaN,5700 S CICERO AVE,CHICAGO,IL,60638.0,03/03/2026,License,Not Ready,NaN,41.789329,-87.741646,"(41.789329323265385, -87.74164564419637)",Restaurant
4467,2629027,SIP & SAVOR COFFEE HOUSE,SIP & SAVOR COFFEE HOUSE,3006813.0,Restaurant,NaN,230 W MONROE ST,CHICAGO,IL,60606.0,12/24/2025,License,Not Ready,NaN,41.880757,-87.634709,"(41.88075715864721, -87.6347092983425)",Restaurant
7807,2625649,THE POINT NIGHTCLUB,THE POINT VENUE,0.0,Unknown,NaN,1565 N MILWAUKEE AVE,CHICAGO,IL,60622.0,10/20/2025,Complaint,No Entry,NaN,41.910035,-87.676705,"(41.91003508427626, -87.67670468869552)",Unknown
9146,2624283,59TH MINI MART INC,59TH MINI MART INC,3051388.0,Convenience Store,NaN,2712 W 59TH ST,CHICAGO,IL,60629.0,09/24/2025,License,Not Ready,NaN,41.786547,-87.691769,"(41.78654714695912, -87.6917689195663)",Convenience Store
14513,2618841,LAVANDERIA AND TIENDITA,LAVANDERIA AND TIENDITA,3031353.0,Unknown,NaN,2800 S TRIPP AVE,CHICAGO,IL,60623.0,06/09/2025,License,Not Ready,NaN,41.840397,-87.730675,"(41.84039653671148, -87.73067501245995)",Unknown
14971,2618413,309 GROCERY & DELI LLC,309 GROCERY & DELI LLC,3002010.0,Grocery Store,NaN,309 S CICERO,CHICAGO,IL,60644.0,06/02/2025,License,Not Ready,NaN,41.876496,-87.744986,"(41.87649563083872, -87.74498590489245)",Grocery Store
23369,2609999,SIP & SAVOR COFFEE HOUSE,SIP & SAVOR COFFEE HOUSE,3006813.0,Restaurant,NaN,230 W MONROE ST,CHICAGO,IL,60606.0,01/03/2025,License,Not Ready,NaN,41.880757,-87.634709,"(41.88075715864721, -87.6347092983425)",Restaurant
29641,2602638,KIDS CAFE 1 & 2,KIDS CAFE 1 & 2,0.0,After School Program,NaN,1060 E 47TH ST,CHICAGO,IL,60653.0,09/12/2024,Canvass,Out of Business,NaN,41.809722,-87.599389,"(41.80972238714215, -87.59938918300288)",After School Program
33342,2596939,HUDSON,HUDSON,2977248.0,Grocery Store,NaN,5700 S CICERO AVE,CHICAGO,IL,60638.0,07/02/2024,License,Not Ready,NaN,41.789329,-87.741646,"(41.789329323265385, -87.74164564419637)",Grocery Store


It appears that 'Risk' is null whereever 'Violations' is null. We will address 'Violations' later, but we can assume for now that a null value for 'Violations' represents literally not having a prior violation, which is valuable information for our analysis. We can validate this assumption before making changes to the data. 

In [77]:
# Find rows where 'Risk' is null and 'Violations' are not 
print(len(food.loc[(food['Risk'].isna()) & (food['Violations'].isna())]))
food.loc[(food['Risk'].isna()) & (food['Violations'].isna() == False)]

85


,Inspection ID,DBA Name,AKA Name,License #,Facility Type,Risk,Address,City,State,Zip,Inspection Date,Inspection Type,Results,Violations,Latitude,Longitude,Location,Facility Type Grouped
251560,1138786,FOODA,FOODA,0.0,Illegal Vendor,NaN,321 N CLARK ST,CHICAGO,IL,60654.0,01/25/2013,Complaint,Fail,12. HAND WASHING FACILITIES: WITH SOAP AND SAN...,41.888087,-87.630888,"(41.88808735310137, -87.6308880745296)",Illegal Vendor
294980,414002,KIDS CAFE 1 & 2,KIDS CAFE 1 & 2,0.0,After School Program,NaN,1060 E 47TH ST,CHICAGO,IL,60653.0,09/29/2010,Consultation,Pass,38. VENTILATION: ROOMS AND EQUIPMENT VENTED AS...,41.809722,-87.599389,"(41.80972238714215, -87.59938918300288)",After School Program


There are only two rows where 'Risk' and 'Violations' are not equal. This could represent first time inspections or violations, so we will still maintain this data. To address the null values in 'Risk', we can repeat the method used for 'Facility Type' and fill null values with a string containing 'Unknown'. 

In [78]:
# Fill null values in 'Risk' column with a string containing 'Unknown'
food['Risk'] = food['Risk'].fillna('Unknown')

# Validate that cleaning was successful 
food.isna().sum()

Inspection ID                0
DBA Name                     0
AKA Name                     0
License #                    0
Facility Type                0
Risk                         0
Address                      0
City                       189
State                       67
Zip                         42
Inspection Date              0
Inspection Type              1
Results                      0
Violations               86553
Latitude                  1030
Longitude                 1030
Location                  1030
Facility Type Grouped        0
dtype: int64

### 'City', 'State', and 'Zip' Columns

Fortunately for this process, the 'Address' column contains no null values, so we can use this to help pinpoint the city and state for rows where they are unknown. Once city and state are established, it will be much easier to locate zip code as well. 

First, we can check the 'City' column to see if null values follow an obvious trend. 

In [79]:
# Print 10 rows where 'City' is null
food.loc[food['City'].isna()].head()

,Inspection ID,DBA Name,AKA Name,License #,Facility Type,Risk,Address,City,State,Zip,Inspection Date,Inspection Type,Results,Violations,Latitude,Longitude,Location,Facility Type Grouped
100,2633593,BON APPETIT AT OBAMA PRESIDENTIAL CENTER,OBAMA PRESIDENTIAL CENTER,3069628.0,Restaurant,Risk 3 (Low),6011 S STONY ISLAND AVE,NaN,IL,60637.0,04/07/2026,License,Pass,NaN,41.785749,-87.586438,"(41.78574881274826, -87.58643828072859)",Restaurant
106,2633595,BON APPETIT AT OBAMA PRESIDENTIAL CENTER,OBAMA PRESIDENTIAL CENTER,3069627.0,Restaurant,Risk 1 (High),6011 S STONY ISLAND AVE,NaN,IL,60637.0,04/07/2026,License,Pass,58. ALLERGEN TRAINING AS REQUIRED - Comments: ...,41.785749,-87.586438,"(41.78574881274826, -87.58643828072859)",Restaurant
113,2633617,BON APPETIT AT OBAMA PRESIDENTIAL CENTER,OBAMA PRESIDENTIAL CENTER,3069633.0,Restaurant,Risk 1 (High),6011 S STONY ISLAND AVE,NaN,IL,60637.0,04/07/2026,License,Pass w/ Conditions,47. FOOD & NON-FOOD CONTACT SURFACES CLEANABLE...,41.785749,-87.586438,"(41.78574881274826, -87.58643828072859)",Restaurant
116,2633621,BON APPETIT AT OBAMA PRESIDENTIAL CENTER,OBAMA PRESIDENTIAL CENTER,3069635.0,Restaurant,Risk 3 (Low),6011 S STONY ISLAND AVE,NaN,IL,60637.0,04/07/2026,License,Pass w/ Conditions,NaN,41.785749,-87.586438,"(41.78574881274826, -87.58643828072859)",Restaurant
125,2633597,BON APPETIT AT OBAMA PRESIDENTIAL CENTER,OBAMA PRESIDENTIAL CENTER,3069630.0,Restaurant,Risk 3 (Low),6011 S STONY ISLAND AVE,NaN,IL,60637.0,04/07/2026,License,Pass,NaN,41.785749,-87.586438,"(41.78574881274826, -87.58643828072859)",Restaurant


This tells us that not all rows which have null 'City' values also have null values for 'Latitude','Longitude', and 'State'. Using this information, we can see how many rows have null values for both 'City' and 'State', and how many unique state names exist in the dataset. 

In [80]:
# Find number of rows where 'City' and 'State' are both null
print(len(food.loc[(food['City'].isna()) & (food['State']).isna()]))

24


In [81]:
# Find unique state names in the dataset
food['State'].unique()

array(['IL', 'CA', nan, 'IN', 'WI', 'CO', 'NY'], dtype=object)

We are only interested in locations in the Chicago area, so we can check that all locations with 'City' values containing 'Chicago' are recorded as inside of Illinois. 

In [82]:
# Count the states that Chicago is recorded being inside 
print(len(food[food['City'].str.contains("Chicago", na = False, case = False)]['State'].unique()))
food[food['City'].str.contains("Chicago", na = False, case = False)]['State'].unique()

2


array(['IL', nan], dtype=object)

Since Chicago is only found in rows where 'State' is either 'IL' or null, we can remove rows where state is not Illinois. 

In [83]:
# Create a list of other state names to avoid dropping null values 
wrong_states = ["CA", "IN", "WI", "CO", "NY"]

# Drop rows where 'State' is in 'wrong_states'
food = food[~food['State'].isin(wrong_states)]

# Validate that cleaning was accurate 
food['State'].unique()

array(['IL', nan], dtype=object)

In [84]:
# Check number of null values in the data set
food.isna().sum()

Inspection ID                0
DBA Name                     0
AKA Name                     0
License #                    0
Facility Type                0
Risk                         0
Address                      0
City                       189
State                       67
Zip                         42
Inspection Date              0
Inspection Type              1
Results                      0
Violations               86547
Latitude                  1010
Longitude                 1010
Location                  1010
Facility Type Grouped        0
dtype: int64

Since there are only 24 rows where 'City' and 'State' are both null, we can use each column to fill the other. 

In [85]:
# Fill 'State' null values with 'IL' if 'City' is 'Chicago'
food.loc[food['City'] == 'Chicago', 'State'] = 'IL'

Now, we need to check the names of the unique cities within Illinois in our dataset. This can tell us how much of our data from Illinois is from Chicago or other areas in the state. 

In [86]:
# Print unique names of cities in Illinois 
food[food['State'] == 'IL']['City'].unique()[:20]

array(['CHICAGO', nan, 'Chicago', 'chicago', 'BERWYN', 'CCHICAGO',
       'SKOKIE', 'CH', 'CHICAGOO', '312CHICAGO', 'CHICAGOCHICAGO',
       'CHicago', 'MOUNT PROSPECT', 'NAPERVILLE', 'EVANSTON', 'CHICAGO.',
       'OAK PARK', 'GRAYSLAKE', 'BROOKFIELD', 'BURBANK'], dtype=object)

Clearly, there is a lot of variation in the way that 'Chicago' was reported in this column. We will normalize the different inputs of 'Chicago' and run the query again. 

In [87]:
# Create list of instances of improper formatting 
chicago_names = ['Chicago', 'Chicago.', 'Chcicago', 'Cchicago', 'chicagoo', 'chicagoi', 'chicagoBedford']

# Convert to single string 
name_filter = "|".join(chicago_names)

apply_name = food['City'].str.contains(name_filter, case = False, na = False)

# Assign 'City' to 'Chicago' where it matches the list of names 
food.loc[apply_name, 'City'] = 'Chicago'

# Change 'City' column to title formatting for consistency
food['City'] = food['City'].str.title()

# Print unique names of cities in Illinois 
food[food['State'] == 'IL']['City'].unique()[:20]

array(['Chicago', nan, 'Berwyn', 'Skokie', 'Ch', 'Mount Prospect',
       'Naperville', 'Evanston', 'Oak Park', 'Grayslake', 'Brookfield',
       'Burbank', 'Matteson', 'Western Springs', 'Plainfield',
       'Highland Park', 'Schaumburg', 'Summit', 'Lake Zurich',
       'Glen Ellyn'], dtype=object)

In [88]:
# Count the number of total rows where 'City' is 'Chicago'
print("Number of rows in Chicago: " + str(len(food.loc[food['City'] == 'Chicago'])))

# Count the number of total rows where 'City' is not 'Chicago'
print("Number of rows outside of Chicago: " + str(len(food.loc[food['City'] != 'Chicago'])))

Number of rows in Chicago: 308131
Number of rows outside of Chicago: 429


Including null values, there are only 429 rows in our dataset that are not found in Chicago. This is only about 0.01% of our data, so we can drop these rows from our dataset without the concern that valuable information would be lost. 

In [89]:
# Drop rows where 'City' is not 'Chicago'
food = food[food['City'] == 'Chicago']

# Validate that cleaning was accurate 
# Count the number of total rows where 'City' is 'Chicago'
print("Number of rows in Chicago: " + str(len(food.loc[food['City'] == 'Chicago'])))

# Count the number of total rows where 'City' is not 'Chicago'
print("Number of rows outside of Chicago: " + str(len(food.loc[food['City'] != 'Chicago'])))

Number of rows in Chicago: 308131
Number of rows outside of Chicago: 0


In [90]:
# Print count of null values 
food.isna().sum()

Inspection ID                0
DBA Name                     0
AKA Name                     0
License #                    0
Facility Type                0
Risk                         0
Address                      0
City                         0
State                       43
Zip                          3
Inspection Date              0
Inspection Type              1
Results                      0
Violations               86296
Latitude                   790
Longitude                  790
Location                   790
Facility Type Grouped        0
dtype: int64

Since we have limited our dataset to rows with Chicago data, we can also confidently assign null values in 'State' to 'IL', for Illinois. 

In [91]:
# Assign null values in 'State' to 'IL'
food['State'] = food['State'].fillna("IL")

# Check null values 
food.isna().sum()

Inspection ID                0
DBA Name                     0
AKA Name                     0
License #                    0
Facility Type                0
Risk                         0
Address                      0
City                         0
State                        0
Zip                          3
Inspection Date              0
Inspection Type              1
Results                      0
Violations               86296
Latitude                   790
Longitude                  790
Location                   790
Facility Type Grouped        0
dtype: int64

For the Zip code, we can again check the rows where 'Zip' is null to look for a pattern. 

In [92]:
# Print rows with missing 'Zip' value
food.loc[food['Zip'].isna()]

,Inspection ID,DBA Name,AKA Name,License #,Facility Type,Risk,Address,City,State,Zip,Inspection Date,Inspection Type,Results,Violations,Latitude,Longitude,Location,Facility Type Grouped
227355,1464217,DUNKIN DONUTS,DUNKIN DONUTS,1515116.0,Restaurant,Risk 2 (Medium),7545 N PAULINA ST,Chicago,IL,NaN,04/02/2014,Canvass,Out of Business,NaN,42.019032,-87.673459,"(42.01903180273219, -87.67345866417395)",Restaurant
267035,1106210,DUNKIN DONUTS,DUNKIN DONUTS,1515116.0,Restaurant,Risk 2 (Medium),7545 N PAULINA ST,Chicago,IL,NaN,04/09/2012,Canvass Re-Inspection,Pass,33. FOOD AND NON-FOOD CONTACT EQUIPMENT UTENSI...,42.019032,-87.673459,"(42.01903180273219, -87.67345866417395)",Restaurant
269370,670661,DUNKIN DONUTS,DUNKIN DONUTS,1515116.0,Restaurant,Risk 2 (Medium),7545 N PAULINA ST,Chicago,IL,NaN,02/21/2012,Complaint,Pass w/ Conditions,"6. HANDS WASHED AND CLEANED, GOOD HYGIENIC PRA...",42.019032,-87.673459,"(42.01903180273219, -87.67345866417395)",Restaurant


Luckily, all three instances share a location. We can use the 'Address' value to find the zip code of this location. 

According to Google Maps, the zip code for this address is 60626, so we can fill the null values with 60626 without concern.

In [93]:
# Identify data type of 'Zip' for consistency 
food['Zip'].info()

<class 'pandas.core.series.Series'>
Index: 308131 entries, 0 to 308584
Series name: Zip
Non-Null Count   Dtype  
--------------   -----  
308128 non-null  float64
dtypes: float64(1)
memory usage: 4.7 MB


In [94]:
# Assign float 60626 to 'Zip' code where there is no value
food['Zip'] = food['Zip'].fillna(float(60626))

# Validate cleaning was successful
food.isna().sum()

Inspection ID                0
DBA Name                     0
AKA Name                     0
License #                    0
Facility Type                0
Risk                         0
Address                      0
City                         0
State                        0
Zip                          0
Inspection Date              0
Inspection Type              1
Results                      0
Violations               86296
Latitude                   790
Longitude                  790
Location                   790
Facility Type Grouped        0
dtype: int64

### 'Inspection Type' and 'Violations' Columns

In [95]:
# Print first five rows from dataset 
food.head()

,Inspection ID,DBA Name,AKA Name,License #,Facility Type,Risk,Address,City,State,Zip,Inspection Date,Inspection Type,Results,Violations,Latitude,Longitude,Location,Facility Type Grouped
0,2634806,ONE STOP FOOD & LIQUOR STORE,ONE STOP FOOD & LIQUOR STORE,1094.0,Grocery Store,Risk 2 (Medium),4301-4323 S LAKE PARK AVE,Chicago,IL,60653.0,04/10/2026,Complaint,Fail,16. FOOD-CONTACT SURFACES: CLEANED & SANITIZED...,41.816865,-87.598689,"(41.816865148052045, -87.59868884416672)",Grocery Store
1,2634820,MR QUILES MEXICAN FOOD #2,MR QUILES MEXICAN FOOD #2,2385750.0,Restaurant,Risk 2 (Medium),2300 S THROOP ST,Chicago,IL,60608.0,04/10/2026,Canvass,Pass,47. FOOD & NON-FOOD CONTACT SURFACES CLEANABLE...,41.850451,-87.658798,"(41.85045102427, -87.65879785567869)",Mobile Food Preparer
2,2634815,LA GUERITA MEXICAN SHACK,LA GUERITA MEXICAN SHACK,2840820.0,Restaurant,Risk 2 (Medium),2300 S THROOP ST,Chicago,IL,60608.0,04/10/2026,Canvass,Pass,NaN,41.850451,-87.658798,"(41.85045102427, -87.65879785567869)",Mobile Food Preparer
3,2634821,THE BBQ SHACK,THE BBQ SHACK,3065284.0,Restaurant,Risk 1 (High),1709 W WASHINGTON BLVD,Chicago,IL,60612.0,04/10/2026,License,Fail,16. FOOD-CONTACT SURFACES: CLEANED & SANITIZED...,41.883143,-87.669767,"(41.88314294811615, -87.66976710838105)",Restaurant
4,2634811,"Thomas, Velma ECC","Thomas, Velma ECC",26891.0,School,Risk 1 (High),3625 S Hoyne ST,Chicago,IL,60609.0,04/10/2026,Canvass,Pass,10. ADEQUATE HANDWASHING SINKS PROPERLY SUPPLI...,41.827769,-87.677505,"(41.82776913597991, -87.67750501083977)",School


Judging from the output of the *head()* function, there does not seem to be direct causation between 'Inspection Type', 'Results', and 'Violations'. However, it does show us what we had assumed earlier: that null values in the 'Violations' column represent a lack of violation, rather than a lack of data. It appears that locations can pass inspections while still violating certain policies. Also, 'Inspection Type' can vary, but seems to be uniform in its formatting. There is only one null value in 'Inspection Type', so we will drop this row.

First, can check the unique values of 'Inspection Type' to see if there is any formatting that should be addressed.

In [96]:
# Print unique values of 'Inspection Type'
food['Inspection Type'].unique()[:10]

array(['Complaint', 'Canvass', 'License', 'Short Form Complaint',
       'Complaint Re-Inspection', 'Non-Inspection',
       'Canvass Re-Inspection', 'License Re-Inspection', 'Not Ready',
       'Recent Inspection'], dtype=object)

There appears to be a lack of normal capitalization, so we will fix that with the *.str.title()* function

In [97]:
# Fix capitalization errors in 'Inspection Type'
food['Inspection Type'] = food['Inspection Type'].str.title()

# Drop row with null value in 'Inspection Type' 
food = food.dropna(subset="Inspection Type")

# Rerun query to check for unique values in 'Inspection Type' 
food['Inspection Type'].unique()[:10]

array(['Complaint', 'Canvass', 'License', 'Short Form Complaint',
       'Complaint Re-Inspection', 'Non-Inspection',
       'Canvass Re-Inspection', 'License Re-Inspection', 'Not Ready',
       'Recent Inspection'], dtype=object)

In [98]:
# Check for null values
food.isna().sum()

Inspection ID                0
DBA Name                     0
AKA Name                     0
License #                    0
Facility Type                0
Risk                         0
Address                      0
City                         0
State                        0
Zip                          0
Inspection Date              0
Inspection Type              0
Results                      0
Violations               86295
Latitude                   790
Longitude                  790
Location                   790
Facility Type Grouped        0
dtype: int64

Now, for the 'Violations' column, we wish to represent a lack of violations in a string format, rather than a null value. We can do this by assigning null values in 'Violations' to a string containing 'None', which will eliminate the concern of null values interfering with analysis. First, we will remove null values in rows with failed inspections, as they would misrepresent the meaning of '0' violations in further analysis.

In [99]:
# Count the number of rows where 'Violations' is null and 'Results' is 'Fail'
print(food['Inspection ID'].loc[(food['Violations'].isna()) & (food['Results'] == 'Fail')].count())

# Print current number of total rows
print(len(food))

3593
308130


Right now, only about 1% of our data is made up of failed inspections with missing violation reports. This is small enough of a percentage to remove these rows entirely. 

In [100]:
# Remove rows where 'Results' is 'Fail' and 'Violations' is null
food = food[~((food['Results'] == 'Fail') & (food['Violations'].isna()))]

# Count the number of rows where 'Violations' is null and 'Results' is 'Fail'
print(food['Inspection ID'].loc[(food['Violations'].isna()) & (food['Results'] == 'Fail')].count())

0


In [101]:
# Assign null values in 'Violations' to 'None' 
food['Violations'] = food['Violations'].fillna("None")

# Validate null value count
food.isna().sum()

Inspection ID              0
DBA Name                   0
AKA Name                   0
License #                  0
Facility Type              0
Risk                       0
Address                    0
City                       0
State                      0
Zip                        0
Inspection Date            0
Inspection Type            0
Results                    0
Violations                 0
Latitude                 781
Longitude                781
Location                 781
Facility Type Grouped      0
dtype: int64

### 'Latitude', 'Longitude', and 'Location' Columns

In [102]:
# Print 10 rows where 'Latitude' is null 
food.loc[food['Latitude'].isna()].head()

,Inspection ID,DBA Name,AKA Name,License #,Facility Type,Risk,Address,City,State,Zip,Inspection Date,Inspection Type,Results,Violations,Latitude,Longitude,Location,Facility Type Grouped
123,2633607,OBAMA PRESIDENTIAL CENTER-TEACHING KITCHEN AND...,OBAMA PRESIDENTIAL CENTER-TEACHING KITCHEN AND...,3078143.0,Restaurant,Risk 2 (Medium),6021 S STONY AVE BLDG,Chicago,IL,60637.0,04/07/2026,License,Fail,"1. PERSON IN CHARGE PRESENT, DEMONSTRATES KNOW...",NaN,NaN,NaN,Restaurant
326,2633281,OSO & THE BULL,OSO & THE BULL,2699078.0,Restaurant,Risk 1 (High),2009 S LAFIN ST,Chicago,IL,60608.0,03/30/2026,Non-Inspection,No Entry,None,NaN,NaN,NaN,Restaurant
955,2632560,DANIEL WILLIAM HALE,DANIEL WILLIAM HALE,66311.0,School,Risk 1 (High),4934 S Wabash (45E) (C/Shabazz),Chicago,IL,60615.0,03/16/2026,Canvass,Pass,None,NaN,NaN,NaN,School
1625,2631858,Sauganash Elementary School,Sauganash Elementary School,25211.0,School,Risk 1 (High),6040 N Kilpatrick (4700W) AVE,Chicago,IL,60646.0,02/26/2026,Canvass,Pass,None,NaN,NaN,NaN,School
1947,2631534,MORRILL,MORRILL,24571.0,School,Risk 1 (High),6011 S Rockwell (2600W) AVE,Chicago,IL,60629.0,02/20/2026,Canvass,Pass w/ Conditions,2. CITY OF CHICAGO FOOD SERVICE SANITATION CER...,NaN,NaN,NaN,School


In [103]:
# Print the number of unique names where 'Latitude' is null
print(len(food['DBA Name'].loc[food['Latitude'].isna()].unique()))

87


In [104]:
# Fill null values based on existing Latitude values
food['Latitude'] = food['Latitude'].fillna(
    food.groupby('DBA Name')['Latitude'].transform('first')
)

# Validate null value count
print(food['Latitude'].isna().sum())

631


In [105]:
# Fill null values based on existing 'Longitude' values
food['Longitude'] = food['Longitude'].fillna(
    food.groupby('DBA Name')['Longitude'].transform('first')
)

# Fill null values based on existing 'Location' values
food['Location'] = food['Location'].fillna(
    food.groupby('DBA Name')['Location'].transform('first')
)

# Validate null values 
food.isna().sum()

Inspection ID              0
DBA Name                   0
AKA Name                   0
License #                  0
Facility Type              0
Risk                       0
Address                    0
City                       0
State                      0
Zip                        0
Inspection Date            0
Inspection Type            0
Results                    0
Violations                 0
Latitude                 631
Longitude                631
Location                 631
Facility Type Grouped      0
dtype: int64

In [106]:
# Print the number of unique names where 'Latitude' is null
print(len(food['DBA Name'].loc[food['Latitude'].isna()].unique()))

66


In [107]:
# Fill null values based on existing 'Latitude' values at matching 'Address' values
food['Latitude'] = food['Latitude'].fillna(
    food.groupby('Address')['Latitude'].transform('first')
)

# Fill null values based on existing 'Longitude' values at matching 'Address' values
food['Longitude'] = food['Longitude'].fillna(
    food.groupby('Address')['Longitude'].transform('first')
)

# Fill null values based on existing 'Location' values at matching 'Address' values
food['Location'] = food['Location'].fillna(
    food.groupby('Address')['Location'].transform('first')
)

# Validate null value count
food.isna().sum()

Inspection ID              0
DBA Name                   0
AKA Name                   0
License #                  0
Facility Type              0
Risk                       0
Address                    0
City                       0
State                      0
Zip                        0
Inspection Date            0
Inspection Type            0
Results                    0
Violations                 0
Latitude                 628
Longitude                628
Location                 628
Facility Type Grouped      0
dtype: int64

In [108]:
# Create variable to represent number of null values in 'Latitude', 'Longitude', 'Latitude'
n_null_loc = 636 

# Variable to represent total number of rows in the dataset 
n = len(food)

# Display the number of null values out of the total dataset
print(f"Number of rows with null coordinate values: {n_null_loc}")
print(f"Total number of rows in dataset: {n}")
print(f"Proportion of dataset: {round((n_null_loc / n)*100, 2)}%")

Number of rows with null coordinate values: 636
Total number of rows in dataset: 304537
Proportion of dataset: 0.21%


In [109]:
# Drop rows with null values for Longtiude/Latitude/Locaiton 
food = food.dropna(subset='Longitude')

# Validate cleaning was successful
food.isna().sum()


Inspection ID            0
DBA Name                 0
AKA Name                 0
License #                0
Facility Type            0
Risk                     0
Address                  0
City                     0
State                    0
Zip                      0
Inspection Date          0
Inspection Type          0
Results                  0
Violations               0
Latitude                 0
Longitude                0
Location                 0
Facility Type Grouped    0
dtype: int64

## Formatting Handling

To handle remaining formatting issues, we can go column by column to see if there are any columns with inconsistent inputs. We know that 'DBA Name' and 'AKA Name' are unique columns, so we are unable to know for sure which rows may have ben inputted incorrectly. However, we can use the *.str.title()* function to change entries from fully capitalized to only having the first letter of each word capitalized. 

In [110]:
# Use .str.title() on 'DBA Name' and 'AKA Name' 
food['DBA Name'] = food['DBA Name'].str.title()
food['AKA Name'] = food['AKA Name'].str.title()

# Print first five rows to ensure cleaning was accurate 
food.head()

,Inspection ID,DBA Name,AKA Name,License #,Facility Type,Risk,Address,City,State,Zip,Inspection Date,Inspection Type,Results,Violations,Latitude,Longitude,Location,Facility Type Grouped
0,2634806,One Stop Food & Liquor Store,One Stop Food & Liquor Store,1094.0,Grocery Store,Risk 2 (Medium),4301-4323 S LAKE PARK AVE,Chicago,IL,60653.0,04/10/2026,Complaint,Fail,16. FOOD-CONTACT SURFACES: CLEANED & SANITIZED...,41.816865,-87.598689,"(41.816865148052045, -87.59868884416672)",Grocery Store
1,2634820,Mr Quiles Mexican Food #2,Mr Quiles Mexican Food #2,2385750.0,Restaurant,Risk 2 (Medium),2300 S THROOP ST,Chicago,IL,60608.0,04/10/2026,Canvass,Pass,47. FOOD & NON-FOOD CONTACT SURFACES CLEANABLE...,41.850451,-87.658798,"(41.85045102427, -87.65879785567869)",Mobile Food Preparer
2,2634815,La Guerita Mexican Shack,La Guerita Mexican Shack,2840820.0,Restaurant,Risk 2 (Medium),2300 S THROOP ST,Chicago,IL,60608.0,04/10/2026,Canvass,Pass,None,41.850451,-87.658798,"(41.85045102427, -87.65879785567869)",Mobile Food Preparer
3,2634821,The Bbq Shack,The Bbq Shack,3065284.0,Restaurant,Risk 1 (High),1709 W WASHINGTON BLVD,Chicago,IL,60612.0,04/10/2026,License,Fail,16. FOOD-CONTACT SURFACES: CLEANED & SANITIZED...,41.883143,-87.669767,"(41.88314294811615, -87.66976710838105)",Restaurant
4,2634811,"Thomas, Velma Ecc","Thomas, Velma Ecc",26891.0,School,Risk 1 (High),3625 S Hoyne ST,Chicago,IL,60609.0,04/10/2026,Canvass,Pass,10. ADEQUATE HANDWASHING SINKS PROPERLY SUPPLI...,41.827769,-87.677505,"(41.82776913597991, -87.67750501083977)",School


We can repeat this process for the 'Address' column, since it appears some addresses are fully capitalized and some are already in title formatting.

In [111]:
# Use .str.title() on 'Address' column
food['Address'] = food['Address'].str.title()

# Print first five rows to ensure cleaning was accurate 
food.head()

,Inspection ID,DBA Name,AKA Name,License #,Facility Type,Risk,Address,City,State,Zip,Inspection Date,Inspection Type,Results,Violations,Latitude,Longitude,Location,Facility Type Grouped
0,2634806,One Stop Food & Liquor Store,One Stop Food & Liquor Store,1094.0,Grocery Store,Risk 2 (Medium),4301-4323 S Lake Park Ave,Chicago,IL,60653.0,04/10/2026,Complaint,Fail,16. FOOD-CONTACT SURFACES: CLEANED & SANITIZED...,41.816865,-87.598689,"(41.816865148052045, -87.59868884416672)",Grocery Store
1,2634820,Mr Quiles Mexican Food #2,Mr Quiles Mexican Food #2,2385750.0,Restaurant,Risk 2 (Medium),2300 S Throop St,Chicago,IL,60608.0,04/10/2026,Canvass,Pass,47. FOOD & NON-FOOD CONTACT SURFACES CLEANABLE...,41.850451,-87.658798,"(41.85045102427, -87.65879785567869)",Mobile Food Preparer
2,2634815,La Guerita Mexican Shack,La Guerita Mexican Shack,2840820.0,Restaurant,Risk 2 (Medium),2300 S Throop St,Chicago,IL,60608.0,04/10/2026,Canvass,Pass,None,41.850451,-87.658798,"(41.85045102427, -87.65879785567869)",Mobile Food Preparer
3,2634821,The Bbq Shack,The Bbq Shack,3065284.0,Restaurant,Risk 1 (High),1709 W Washington Blvd,Chicago,IL,60612.0,04/10/2026,License,Fail,16. FOOD-CONTACT SURFACES: CLEANED & SANITIZED...,41.883143,-87.669767,"(41.88314294811615, -87.66976710838105)",Restaurant
4,2634811,"Thomas, Velma Ecc","Thomas, Velma Ecc",26891.0,School,Risk 1 (High),3625 S Hoyne St,Chicago,IL,60609.0,04/10/2026,Canvass,Pass,10. ADEQUATE HANDWASHING SINKS PROPERLY SUPPLI...,41.827769,-87.677505,"(41.82776913597991, -87.67750501083977)",School


We can see the current data type for each column to assess whether they are correct.

Currently, 'License #' and 'Zip' are encoded as floats, which is unncessecary since neither variable can contain decimal values. We can convert these two columns to integers to avoid having unneeded floating point values. 

In [112]:
# Convert 'License #' and 'Zip' to integers
food['License #'] = food['License #'].astype(int)
food['Zip'] = food['Zip'].astype(int)

food.head()

,Inspection ID,DBA Name,AKA Name,License #,Facility Type,Risk,Address,City,State,Zip,Inspection Date,Inspection Type,Results,Violations,Latitude,Longitude,Location,Facility Type Grouped
0,2634806,One Stop Food & Liquor Store,One Stop Food & Liquor Store,1094,Grocery Store,Risk 2 (Medium),4301-4323 S Lake Park Ave,Chicago,IL,60653,04/10/2026,Complaint,Fail,16. FOOD-CONTACT SURFACES: CLEANED & SANITIZED...,41.816865,-87.598689,"(41.816865148052045, -87.59868884416672)",Grocery Store
1,2634820,Mr Quiles Mexican Food #2,Mr Quiles Mexican Food #2,2385750,Restaurant,Risk 2 (Medium),2300 S Throop St,Chicago,IL,60608,04/10/2026,Canvass,Pass,47. FOOD & NON-FOOD CONTACT SURFACES CLEANABLE...,41.850451,-87.658798,"(41.85045102427, -87.65879785567869)",Mobile Food Preparer
2,2634815,La Guerita Mexican Shack,La Guerita Mexican Shack,2840820,Restaurant,Risk 2 (Medium),2300 S Throop St,Chicago,IL,60608,04/10/2026,Canvass,Pass,None,41.850451,-87.658798,"(41.85045102427, -87.65879785567869)",Mobile Food Preparer
3,2634821,The Bbq Shack,The Bbq Shack,3065284,Restaurant,Risk 1 (High),1709 W Washington Blvd,Chicago,IL,60612,04/10/2026,License,Fail,16. FOOD-CONTACT SURFACES: CLEANED & SANITIZED...,41.883143,-87.669767,"(41.88314294811615, -87.66976710838105)",Restaurant
4,2634811,"Thomas, Velma Ecc","Thomas, Velma Ecc",26891,School,Risk 1 (High),3625 S Hoyne St,Chicago,IL,60609,04/10/2026,Canvass,Pass,10. ADEQUATE HANDWASHING SINKS PROPERLY SUPPLI...,41.827769,-87.677505,"(41.82776913597991, -87.67750501083977)",School


The output of the *.info()* function also shows us that the 'Inspection Date' column is encoded as a string, rather than a pandas datetime type. We can convert this column while maintaining the layout from the original data. 

In [113]:
# Convert 'Inspection Date' to pandas datetime objects
food['Inspection Date'] = pd.to_datetime(food['Inspection Date']) 

# View first five rows
food.head()

,Inspection ID,DBA Name,AKA Name,License #,Facility Type,Risk,Address,City,State,Zip,Inspection Date,Inspection Type,Results,Violations,Latitude,Longitude,Location,Facility Type Grouped
0,2634806,One Stop Food & Liquor Store,One Stop Food & Liquor Store,1094,Grocery Store,Risk 2 (Medium),4301-4323 S Lake Park Ave,Chicago,IL,60653,2026-04-10,Complaint,Fail,16. FOOD-CONTACT SURFACES: CLEANED & SANITIZED...,41.816865,-87.598689,"(41.816865148052045, -87.59868884416672)",Grocery Store
1,2634820,Mr Quiles Mexican Food #2,Mr Quiles Mexican Food #2,2385750,Restaurant,Risk 2 (Medium),2300 S Throop St,Chicago,IL,60608,2026-04-10,Canvass,Pass,47. FOOD & NON-FOOD CONTACT SURFACES CLEANABLE...,41.850451,-87.658798,"(41.85045102427, -87.65879785567869)",Mobile Food Preparer
2,2634815,La Guerita Mexican Shack,La Guerita Mexican Shack,2840820,Restaurant,Risk 2 (Medium),2300 S Throop St,Chicago,IL,60608,2026-04-10,Canvass,Pass,None,41.850451,-87.658798,"(41.85045102427, -87.65879785567869)",Mobile Food Preparer
3,2634821,The Bbq Shack,The Bbq Shack,3065284,Restaurant,Risk 1 (High),1709 W Washington Blvd,Chicago,IL,60612,2026-04-10,License,Fail,16. FOOD-CONTACT SURFACES: CLEANED & SANITIZED...,41.883143,-87.669767,"(41.88314294811615, -87.66976710838105)",Restaurant
4,2634811,"Thomas, Velma Ecc","Thomas, Velma Ecc",26891,School,Risk 1 (High),3625 S Hoyne St,Chicago,IL,60609,2026-04-10,Canvass,Pass,10. ADEQUATE HANDWASHING SINKS PROPERLY SUPPLI...,41.827769,-87.677505,"(41.82776913597991, -87.67750501083977)",School


The 'Inspection Type' column should contain very limited values. We can check some of the unique values of this column to ensure that there were no errors with inputting or spelling.

In [114]:
# Check unique values in 'Inspection Type
food['Inspection Type'].unique()[:10]

array(['Complaint', 'Canvass', 'License', 'Short Form Complaint',
       'Complaint Re-Inspection', 'Non-Inspection',
       'Canvass Re-Inspection', 'License Re-Inspection', 'Not Ready',
       'Recent Inspection'], dtype=object)

'Inspection Type' appears uniform, with very precise inputs. Because this precision can benefit our analysis, we will leave this column the way it is. 

We can repeat this process on the 'Results' column, since it should also contain a limited number of possible inputs. 

In [115]:
# Check unique values in 'Results'
food['Results'].unique()

array(['Fail', 'Pass', 'Pass w/ Conditions', 'Not Ready', 'No Entry',
       'Out of Business', 'Business Not Located'], dtype=object)

These inputs appear uniform and they carry specific insights into how locations performed, so we will keep this column the way it is. 

Finally, we can use the *.str.title()* function again on 'Violations' for uniform formatting across the entire dataset. 

In [116]:
# Use .str.title() on 'Violations'
food['Violations'] = food['Violations'].str.title()

# Check first five rows 
food.head()

,Inspection ID,DBA Name,AKA Name,License #,Facility Type,Risk,Address,City,State,Zip,Inspection Date,Inspection Type,Results,Violations,Latitude,Longitude,Location,Facility Type Grouped
0,2634806,One Stop Food & Liquor Store,One Stop Food & Liquor Store,1094,Grocery Store,Risk 2 (Medium),4301-4323 S Lake Park Ave,Chicago,IL,60653,2026-04-10,Complaint,Fail,16. Food-Contact Surfaces: Cleaned & Sanitized...,41.816865,-87.598689,"(41.816865148052045, -87.59868884416672)",Grocery Store
1,2634820,Mr Quiles Mexican Food #2,Mr Quiles Mexican Food #2,2385750,Restaurant,Risk 2 (Medium),2300 S Throop St,Chicago,IL,60608,2026-04-10,Canvass,Pass,47. Food & Non-Food Contact Surfaces Cleanable...,41.850451,-87.658798,"(41.85045102427, -87.65879785567869)",Mobile Food Preparer
2,2634815,La Guerita Mexican Shack,La Guerita Mexican Shack,2840820,Restaurant,Risk 2 (Medium),2300 S Throop St,Chicago,IL,60608,2026-04-10,Canvass,Pass,None,41.850451,-87.658798,"(41.85045102427, -87.65879785567869)",Mobile Food Preparer
3,2634821,The Bbq Shack,The Bbq Shack,3065284,Restaurant,Risk 1 (High),1709 W Washington Blvd,Chicago,IL,60612,2026-04-10,License,Fail,16. Food-Contact Surfaces: Cleaned & Sanitized...,41.883143,-87.669767,"(41.88314294811615, -87.66976710838105)",Restaurant
4,2634811,"Thomas, Velma Ecc","Thomas, Velma Ecc",26891,School,Risk 1 (High),3625 S Hoyne St,Chicago,IL,60609,2026-04-10,Canvass,Pass,10. Adequate Handwashing Sinks Properly Suppli...,41.827769,-87.677505,"(41.82776913597991, -87.67750501083977)",School


In [117]:
# Print the number of rows in the dataset 
print(f"Total number of rows: {len(food)}")
print(f"Out of original {n_before_cleaning} rows.")

Total number of rows: 303909
Out of original 308585 rows.


While creating visualizations, we found that the 'Violations' column contains multiple violations for each inspection, separated by a "|" character. We can create new rows with each of these violations so that we can analyze them further. We can also create a column holding only the violation number, to avoid the uniqueness of added comments. 

Note: pandas online documentation was referred to for this section. The website can be found at:
https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.explode.html 

In [118]:
# Create new column to hold values of 'Violations'
food['Violations List'] = food['Violations'].str.split('|')

# Explode into new rows
food = food.explode('Violations List')

# Clean new rows 
food['Violations List'] = food['Violations List'].str.strip()

# Drop any empty rows
food = food[food['Violations List'] != ""]

In [119]:
# Extract violation number in new column
food['Violation Code'] = food['Violations List'].str.extract(r"(\d+)")

# Test that 0 is not a valid Violation Code
print((food['Violation Code'] == 0).sum())

# Convert null values to 0
food['Violation Code'] = food['Violation Code'].fillna(0)

# Convert to integer 
food['Violation Code'] = food['Violation Code'].astype(int)

# Print first five rows
food.head()

0


,Inspection ID,DBA Name,AKA Name,License #,Facility Type,Risk,Address,City,State,Zip,Inspection Date,Inspection Type,Results,Violations,Latitude,Longitude,Location,Facility Type Grouped,Violations List,Violation Code
0,2634806,One Stop Food & Liquor Store,One Stop Food & Liquor Store,1094,Grocery Store,Risk 2 (Medium),4301-4323 S Lake Park Ave,Chicago,IL,60653,2026-04-10,Complaint,Fail,16. Food-Contact Surfaces: Cleaned & Sanitized...,41.816865,-87.598689,"(41.816865148052045, -87.59868884416672)",Grocery Store,16. Food-Contact Surfaces: Cleaned & Sanitized...,16
0,2634806,One Stop Food & Liquor Store,One Stop Food & Liquor Store,1094,Grocery Store,Risk 2 (Medium),4301-4323 S Lake Park Ave,Chicago,IL,60653,2026-04-10,Complaint,Fail,16. Food-Contact Surfaces: Cleaned & Sanitized...,41.816865,-87.598689,"(41.816865148052045, -87.59868884416672)",Grocery Store,"38. Insects, Rodents, & Animals Not Present - ...",38
0,2634806,One Stop Food & Liquor Store,One Stop Food & Liquor Store,1094,Grocery Store,Risk 2 (Medium),4301-4323 S Lake Park Ave,Chicago,IL,60653,2026-04-10,Complaint,Fail,16. Food-Contact Surfaces: Cleaned & Sanitized...,41.816865,-87.598689,"(41.816865148052045, -87.59868884416672)",Grocery Store,"55. Physical Facilities Installed, Maintained ...",55
1,2634820,Mr Quiles Mexican Food #2,Mr Quiles Mexican Food #2,2385750,Restaurant,Risk 2 (Medium),2300 S Throop St,Chicago,IL,60608,2026-04-10,Canvass,Pass,47. Food & Non-Food Contact Surfaces Cleanable...,41.850451,-87.658798,"(41.85045102427, -87.65879785567869)",Mobile Food Preparer,47. Food & Non-Food Contact Surfaces Cleanable...,47
2,2634815,La Guerita Mexican Shack,La Guerita Mexican Shack,2840820,Restaurant,Risk 2 (Medium),2300 S Throop St,Chicago,IL,60608,2026-04-10,Canvass,Pass,None,41.850451,-87.658798,"(41.85045102427, -87.65879785567869)",Mobile Food Preparer,None,0


In [120]:
import numpy as np

# Assign null values in 'Violations List' to 'None' 
food.loc[food['Violations List'] == np.nan, 'Violations List'] = 'None'

# Assign null values in 'Violaitons Code' to 0
food.loc[food['Violation Code'].isna(), 'Violation Code'] = 0

# Drop 'Violations' column
food = food.drop(columns=['Violations'])

In [121]:
# Count total rows and null values
print("Length of dataset before cleaning: " + str(n_before_cleaning))
print("Length of dataset after cleaning: " + str(len(food)))
food.isna().sum()

Length of dataset before cleaning: 308585
Length of dataset after cleaning: 1082854


Inspection ID            0
DBA Name                 0
AKA Name                 0
License #                0
Facility Type            0
Risk                     0
Address                  0
City                     0
State                    0
Zip                      0
Inspection Date          0
Inspection Type          0
Results                  0
Latitude                 0
Longitude                0
Location                 0
Facility Type Grouped    0
Violations List          0
Violation Code           0
dtype: int64

After cleaning and addressing formatting mistakes, we were left with a dataset containing over one million actionable rows. We can export this dataset to a new csv file called 'food-data-clean' to carry on with visualization and analysis. 

In [122]:
# Create csv file containing clean data 
food.to_csv("../data/processed/food-data-clean.csv", index = False)